# Reinforcement Learning (RL) Env Code
UAV (Unmanned Aerial Vehicle) within ROS (Robot Operating System) environment, using the flightmare simulation framework.

1 UavController Class:
• Controlling UAV within flightmare.
• Initializes ROS nodes, sets up publishers and subscribers for UAV commands (e.g., arm, start, land, move to pose, etc.).
• Handles feedback from the autopilot.
• Control the UAV's velocity.
• Includes callback function (_autopilot_feedback_cb)to update the current pose of UAV based on feedback from the autopilot.

2 Env Class:
• Encapsulates flightmare simulation environment.
• Handles the launching and shutting down of the simulation environment using ROS launch files.
• Provides methods to add a UAV controller, reset the UAV to its initial position, and set up a task trajectory.
• Includes methods to launch and shutdown a SLAM (Simultaneous Localization and Mapping) process.

In [ ]:
# RL Env Encapsulation
import os
import sys
import time
import math
import numpy
import rospy
import signal
import rospkg
import roslaunch
import subprocess
import numpy as np
import geometry_msgs.msg as geometry_msgs
import quadrotor_msgs.msg as quadrotor_msgs
import std_msgs.msg as std_msgs
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from lib.Lie import *

class UavController:
    '''
    UavController class encapsulates UAV controller for flightmare environment.
    Handles ROS node initialization, publishers, subscribers, & UAV control commands.
    '''

    def __init__(self):
        # Initialize ROS node for UAV control
        rospy.init_node('uavController',anonymous=False)
        self._quad_namespace = None
        self._connected = False

        # Initialize publishers for various UAV commands
        self._arm_bridge_pub = None
        self._start_pub = None
        self._land_pub = None
        self._off_pub = None
        self._force_hover_pub = None
        self._go_to_pose_pub = None
        self._set_velo_pub = None

        # Initialize state message and timestamp for autopilot feedback
        self.state_msg = quadrotor_msgs.AutopilotFeedback()
        self._autopilot_feedback_stamp = rospy.Time.now()

        # Store the previous autopilot state
        self._previous_autopilot_state = self.state_msg.OFF

        # Current pose of the UAV
        self.curPose = [0.0,0.0,0.0,0.0]

        # Rate objects for controlling loop frequency
        self.rate50hz = rospy.Rate(50)
        self.rate1hz = rospy.Rate(1)

    def connect(self, quad_namespace):
        # Connect to the UAV namespace and set up publishers and subscribers
        self._quad_namespace = quad_namespace

        self._arm_bridge_pub = rospy.Publisher(
            quad_namespace+'/bridge/arm', std_msgs.Bool, queue_size=1)
        self._start_pub = rospy.Publisher(
            quad_namespace+'/autopilot/start', std_msgs.Empty, queue_size=1)
        self._land_pub = rospy.Publisher(
            quad_namespace+'/autopilot/land', std_msgs.Empty, queue_size=1)
        self._off_pub = rospy.Publisher(
            quad_namespace+'/autopilot/off', std_msgs.Empty, queue_size=1)
        self._force_hover_pub = rospy.Publisher(
            quad_namespace+'/autopilot/force_hover', std_msgs.Empty,
            queue_size=1)
        self._go_to_pose_pub = rospy.Publisher(
            quad_namespace+'/autopilot/pose_command', geometry_msgs.PoseStamped, queue_size=1)
        self._set_velo_pub = rospy.Publisher(
            quad_namespace+'/autopilot/velocity_command', geometry_msgs.TwistStamped, queue_size=50)

        # Subscribe to autopilot feedback
        self._autopilot_feedback_sub = rospy.Subscriber(
            quad_namespace+'/autopilot/feedback',
            quadrotor_msgs.AutopilotFeedback, self._autopilot_feedback_cb)
        self._connected = True

    def _disconnect_pub_sub(self, pub):
        # Disconnect a publisher or subscriber
        if pub is not None:
            pub.unregister()
            pub = None

    def _autopilot_feedback_cb(self, msg):
        # Callback function to update the current pose based on autopilot feedback
        x = msg.reference_state.pose.position.x
        y = msg.reference_state.pose.position.y
        z = msg.reference_state.pose.position.z
        w = msg.reference_state.heading
        self.curPose = [round(x,1),round(y,1),round(z,1),round(w,1)]
        self.state_msg = msg
        self._autopilot_feedback_stamp = rospy.Time.now()

    def disconnect(self):
        # Disconnect all publishers and subscribers
        self._disconnect_pub_sub(self._autopilot_feedback_sub)
        self._disconnect_pub_sub(self._arm_bridge_pub)
        self._disconnect_pub_sub(self._start_pub)
        self._disconnect_pub_sub(self._land_pub)
        self._disconnect_pub_sub(self._off_pub)
        self._disconnect_pub_sub(self._force_hover_pub)
        self._disconnect_pub_sub(self._go_to_pose_pub)
        self._connected = False

    def isConnnect(self):
        # Check if the UAV is connected
        return self._connected

    def arm_bridge(self):
        # Arm the UAV bridge
        arm_message = std_msgs.Bool(True)
        self._arm_bridge_pub.publish(arm_message)

    def start(self):
        # Start the UAV
        start_message = std_msgs.Empty()
        self._start_pub.publish(start_message)

    def land(self):
        # Land the UAV
        land_message = std_msgs.Empty()
        self._land_pub.publish(land_message)

    def powerOff(self):
        # Power off the UAV
        start_message = std_msgs.Empty()
        self._off_pub.publish(start_message)

    def forceHover(self):
        # Force the UAV to hover
        force_hover_msg = std_msgs.Empty()
        self._force_hover_pub.publish(force_hover_msg)

    def to_pose(self,x,y,z,w):
        # Move the UAV to a specific pose
        go_to_pose_msg = geometry_msgs.PoseStamped()
        go_to_pose_msg.pose.position.x = float(x)
        go_to_pose_msg.pose.position.y = float(y)
        go_to_pose_msg.pose.position.z = float(z)

        heading = float(w) / 180.0 * math.pi
        go_to_pose_msg.pose.orientation.w = math.cos(heading / 2.0)
        go_to_pose_msg.pose.orientation.z = math.sin(heading / 2.0)
        self._go_to_pose_pub.publish(go_to_pose_msg)

    def _pub_velo(self,x,y,z,ax,ay,az):
        # Publish velocity commands to the UAV
        set_velo_msg = geometry_msgs.TwistStamped()
        set_velo_msg.twist.linear.x = float(x)
        set_velo_msg.twist.linear.y = float(y)
        set_velo_msg.twist.linear.z = float(z)
        set_velo_msg.twist.angular.x = float(ax);
        set_velo_msg.twist.angular.y = float(ay);
        set_velo_msg.twist.angular.z = float(az);
        self._set_velo_pub.publish(set_velo_msg)

    def move_velo_time(self,x,y,z,ax,ay,az,t):
        # Move the UAV with a specific velocity for a given time duration
        for i in range(int(t*50)):
            self._pub_velo(x,y,z,ax,ay,az);
            self.rate50hz.sleep()

    def dash_to_pose(self,x,y,z):
        # Move the UAV to a specific pose with velocity control
        while(True):
            m = self.state_msg
            clip_lb = 1
            clip_ub = 10
            vx = x - m.reference_state.pose.position.x
            vx = np.clip(vx,clip_lb,clip_ub) if vx>0 else np.clip(vx,-clip_ub,-clip_lb)
            vy = y - m.reference_state.pose.position.y
            vy = np.clip(vy,clip_lb,clip_ub) if vy>0 else np.clip(vy,-clip_ub,-clip_lb)
            vz = z - m.reference_state.pose.position.z
            vz = np.clip(vz,clip_lb,clip_ub) if vz>0 else np.clip(vz,-clip_ub,-clip_lb)

            for i in range(3):
                self._pub_velo(vx,vy,vz,0,0,0);
                self.rate50hz.sleep()

            m = self.state_msg
            if (x - m.reference_state.pose.position.x)*(x - m.reference_state.pose.position.x)+\
                (y - m.reference_state.pose.position.y)*(y - m.reference_state.pose.position.y)+\
                (z - m.reference_state.pose.position.z)*(z - m.reference_state.pose.position.z)< 1:
                break

        while(self.get_autopilot_state_name() != "HOVER"):
            self.rate50hz.sleep()
        self.to_pose(x,y,z,0)

    def get_autopilot_state_name(self):
        # Get the current autopilot state as a string
        if (self.state_msg.autopilot_state == self.state_msg.START):
            return "START"
        if (self.state_msg.autopilot_state == self.state_msg.HOVER):
            return "HOVER"
        if (self.state_msg.autopilot_state == self.state_msg.LAND):
            return "LAND"
        if (self.state_msg.autopilot_state == self.state_msg.EMERGENCY_LAND):
            return "EMERGENCY_LAND"
        if (self.state_msg.autopilot_state == self.state_msg.BREAKING):
            return "BREAKING"
        if (self.state_msg.autopilot_state == self.state_msg.GO_TO_POSE):
            return "GO_TO_POSE"
        if (self.state_msg.autopilot_state == self.state_msg.VELOCITY_CONTROL):
            return "VELOCITY_CONTROL"
        if (self.state_msg.autopilot_state == self.state_msg.REFERENCE_CONTROL):
            return "REFERENCE_CONTROL"
        if (self.state_msg.autopilot_state == self.state_msg.TRAJECTORY_CONTROL):
            return "TRAJECTORY_CONTROL"
        if (self.state_msg.autopilot_state == self.state_msg.COMMAND_FEEDTHROUGH):
            return "COMMAND_FEEDTHROUGH"
        if (self.state_msg.autopilot_state == self.state_msg.RC_MANUAL):
            return "RC_MANUAL"
        return "OFF"

    def imuInitFlightPath(self):
        # Initialize the UAV's flight path using IMU data
        init_rate = rospy.Rate(1/2)
        v = 0.3
        self.move_velo_time(v,v,-v,0,0,0,5)
        init_rate.sleep()
        self.move_velo_time(-v,0,0,0,0,0,8)
        init_rate.sleep()
        self.move_velo_time(v,0,0,0,0,0,3)
        init_rate.sleep()
        v = 0.4
        self.move_velo_time(0,0,v,0,0,0,5)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()

class Env:
    '''
    Env class encapsulates the flightmare simulation environment.
    Handles the launching and shutting down of the simulation environment.
    '''

    def __init__(self):
        # Path to the flightmare environment launch file
        # !Modify the launchPath to your flightmare env launch file.

        self.launchPath = "/home/dbq/dbq/DRL_SLAM/env/flightmare_ws/src/flightmare/flightros/launch/pilot/rotors_gazebo.launch"

        self.uuid = roslaunch.rlutil.get_or_generate_uuid(None, False)
        # Initialize the ROS launch parent

        # roslaunch.parent.ROSLaunchParent.VERBOSE=False
        self.launchHandle = roslaunch.parent.ROSLaunchParent(self.uuid, [self.launchPath],verbose=False)
        self.uav = None
        self.task_trajectory = None
        self.uav_name = "/hummingbird"

        # Initial position of the UAV
        self.initPosition = [0.0,0.0,2.0,0.0]

        # SLAM progress
        self.slamPg = None

    def setup(self):
        # Set up the simulation environment
        roslaunch.configure_logging(self.uuid)
        self.launchHandle.start()

    def add_uavController(self):
        # Add a UAV controller to the environment
        self.uav = UavController()

        if not self.uav.isConnnect():
            self.uav.connect(self.uav_name)
            while not self.uav.isConnnect():
                pass

    def close(self):
        # Shutdown the simulation environment
        self.launchHandle.shutdown()
        self.uuid = roslaunch.rlutil.get_or_generate_uuid(None, False)
        self.launchHandle = roslaunch.parent.ROSLaunchParent(self.uuid, [self.launchPath])

    def reset(self):
        # Reset the UAV to its initial position
        if self.uav.get_autopilot_state_name() == 'OFF':
            rospy.loginfo("uav take-off ...")
            self.uav.rate1hz.sleep()
            self.uav.arm_bridge()
            self.uav.rate1hz.sleep()
            self.uav.start()
            while not self.isReset():
                self.uav.rate50hz.sleep()
            rospy.loginfo("The uav position is initialized successfully!")
        else:
            rospy.loginfo("Move the drone to the initialization position.")
            self.uav.dash_to_pose(0,0,2)
            while not self.isReset():
                self.uav.rate50hz.sleep()
            rospy.loginfo("The uav position is initialized successfully.")

    def reset_centry(self):
        # Reset the UAV to the center position
        self.uav.dash_to_pose(0,0,10)
        while self.uav.curPose != [0.0,0.0,10.0,0.0]:
            self.uav.rate50hz.sleep()

    def set_task_trajectory(self,goals):
        # Set the task trajectory for the UAV
        self.task_trajectory = goals

    def isReset(self):
        # Check if the UAV is reset to the initial position
        return self.uav.curPose == self.initPosition

    def slamLauncher(self):
        # Launch the SLAM process
        # Modify SLAM executable script path like: /home/dbq/dbq/DRL_SLAM/slam/exp/fm_orbslam3/stereoInertial
        self.slamPg = subprocess.Popen("sh run.sh",shell=True,cwd="/home/dbq/dbq/DRL_SLAM/slam/exp/fm_orbslam3/stereoInertial",start_new_session=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)

    def slamShutdown(self):
        # Shutdown the SLAM process
        os.killpg(os.getpgid(self.slamPg.pid),signal.SIGTERM)

In [ ]:
# Checks if GPU is available for use with PyTorch
import torch
print(torch.cuda.is_available())

# Flightmare Simulation Environment

In [ ]:
# Start the flightmare
env = Env()
env.setup()
env.add_uavController()
env.reset()

In [ ]:
# # Close the Flightmare
# env.close()

# PPO Agent - Code Block

# TestEnv
TestEnv class在Env class外又包了一层
作用是测试一些env函数的功能，但不需要每次修改都重启仿真环境

In [ ]:
# Env Test Code block
from concurrent.futures import ThreadPoolExecutor
import threading
from rlslam.msg import pixelstream
import geometry_msgs.msg as geometry_msgs
from geometry_msgs.msg import PointStamped, PoseStamped, TransformStamped

class avgPool10(nn.Module):
    def __init__(self):
        super(avgPool10,self).__init__()
        self.layer1 = nn.AvgPool2d(10,stride=10)
        self.layer2 = nn.MaxPool2d(3,stride=1,padding=1)

    def forward(self, x):
        x = torch.from_numpy(x)
        x = x.unsqueeze(0)
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.unsqueeze(0)
        return x

class avgPool30(nn.Module):
    def __init__(self):
        super(avgPool30,self).__init__()
        self.layer1 = nn.AvgPool2d(30,stride=30)

    def forward(self, x):
        x = torch.from_numpy(x)
        x = x.unsqueeze(0)
        x = self.layer1(x)
        x = torch.flatten(x)
        return x

class TestEnv:
    def __init__(self,renv):
        self.env = renv
        self.avgPL = None
        # step函数连续控制独立线程
        self.stepThread = None
        self.threadPool = ThreadPoolExecutor(max_workers=2)
        self.future = self.threadPool.submit(self.stepVoid)

        # 在线校正
        self.recordAlignTraj = False
        self.alignTraj_ref = []
        self.alignTraj_est = []

        # 原始state 数据
        self.stateDict = {}
        self.last_ftState = None
        self.last_imuState = None

        # 维护 pose_est, pose_ref
        self.P_est = np.zeros(3)
        self.P_est_rot=np.zeros(4)
        self.P_est_ts = 0.0
        self.P_ref = np.zeros(3)#维护的是P_est对应的真值，每次更新P_est时都会更新P_ref
        self.P_ref_ts = 0.0
        self.P_ref_buf = {}
        self.P_gt = np.zeros(3)
        self.P_gt_rot=np.zeros(4)
        self.cbf = False # the clean buffer flag of P_ref_buf
        self.P_ref_buf_recordFlag = True
        self.curRpe = 0

        # 维护 一秒前的Pest Pref ,由于Pest的输出频率为20hz，因此维护一个长度为20的列表，并且设置一个index
        # 这个笔记本训练的policy 基于10hz的slam，步长为0.5秒。因此需要计算0.5s的rpe。window长度为5
        self.rpeWindow = [None for x in range(5)] # change : 10hz
        self.rpeIdx = 0
        # 维护一个逐帧rpe
        self.curFrame_Rpe = 0
        self.RpeFrameNum = 5
        self.cur_ape = 0
        # 维护世界坐标系原点与机坐标系原点
        self.TGU = None

        # 任务结束标志
        self.done = True

        # 奖励函数参数
        # Reward Function E -> [-1,0]
        self.curRe = 0
        self.rpe_mu=0.024
        self.rpe_sigma=0.0045
        self.RE_a = 1.0
        self.RE_b = self.rpe_mu
        self.RE_c = -1

        ## Reward Function G -> [-1,0]
        self.curRg = 0
        self.mid_action = 2.0    #动作中值
        self.action_lb = -0.5   #动作值下边界
        self.action_ub = 0.5    #动作值上边界
        self.action_ex_lb = self.mid_action + self.action_lb #执行动作值上边界
        self.action_ex_ub = self.mid_action + self.action_ub #执行动作值下边界
        ### 简单线性函数
        self.action = 0 # action = actor(state).tanh -> [-1,1]
        self.RG_a = 0.5
        self.RG_b = -0.5

        ## Reward weight
        self.WE = 2 # 2->3
        self.WG = 0.5 # 0.5
        self.deviatedReward = -1000


        ## error reward: 有时rpe会非常大：rpe>5 或者rpe为0 这些经验都不应该加入到buffer里
        self.rewardError = True
        self.rewardErrorCnt = 0
        self.reward = -1

        # slam launcher
        self.isSlamLaunchSucc = False

        # checkErrorBuffer
        self.checkErrorBuffer=[np.zeros(3),np.ones(3),np.zeros(3),np.ones(3),np.zeros(3)]
        self.checkId=0

        self.stateSub = None
        self.poseEstSub = None
        self.poseRefSub = None
        self.slamModePub = None
        self.ReAlignFlagPub = None

    def getRewardE(self):
        """计算误差奖励"""
        arpe = self.curRpe
        RE= self.RE_c*math.tanh(self.exp(self.func(arpe)))
        return RE

    def getRewardG(self):
        """用于计算进度奖励"""
        return self.RG_a*self.action+self.RG_b

    def stateCallback(self,msgs): # 训练一个网络的版本
        if self.stateDict != {} and msgs.imuBiasAx == 0: # imubias 为 0 则不更新
            self.stateDict['timestamp'] = msgs.timestamp
            self.stateDict['pixelNum'] = msgs.pixelNum
            self.stateDict['pixelstring'] = msgs.pixelstring
        else:
            self.stateDict = {
                'timestamp':msgs.timestamp,
                'pixelNum':msgs.pixelNum,
                'imuBiasAx':msgs.imuBiasAx,
                'imuBiasAy':msgs.imuBiasAy,
                'imuBiasAz':msgs.imuBiasAz,
                'imuBiasGx':msgs.imuBiasGx,
                'imuBiasGy':msgs.imuBiasGy,
                'imuBiasGz':msgs.imuBiasGz,
                'Vx':msgs.Vx,
                'Vy':msgs.Vy,
                'Vz':msgs.Vz,
                'pixelstring':msgs.pixelstring
            }

    def getCurState(self,dir):
        # dir = 0:up
        pixList = self.stateDict['pixelstring'].split(';')
        pixList.pop()
        ftState = np.zeros((480,720),np.float32)
        if dir == 0:
            for onepixStr in pixList:#  pix[1]: pt.x 即 col  pix[2]: pt.y 即 row
                pix = onepixStr.split(' ')
                ftState[int(pix[2])][int(pix[1])]=1 #方向向上
        else:
            for onepixStr in pixList:#  pix[1]: pt.x 即 col  pix[2]: pt.y 即 row
                pix = onepixStr.split(' ')
                ftState[479 - int(pix[2])][int(pix[1])]=1 #方向向下
        last_ftState = self.avgPL(ftState).to('cuda:0')
        last_imuState = torch.from_numpy(np.array([
            float(self.action),# 用上一个时刻的动作代替速度
            float(self.action),
            float(self.action),
            float(self.stateDict['imuBiasAx']),
            float(self.stateDict['imuBiasAy']),
            float(self.stateDict['imuBiasAz'])]
        ).astype(np.float32)).unsqueeze(0).to('cuda:0')

        return last_ftState,last_imuState

    def getCurStateOffline(self):
        last_ftState = self.stateDict['pixelstring']

        last_imuState = torch.from_numpy(np.array([
            float(self.action),# 用上一个时刻的动作代替速度
            float(self.action),
            float(self.action),
            float(self.stateDict['imuBiasAx']),
            float(self.stateDict['imuBiasAy']),
            float(self.stateDict['imuBiasAz'])]
        ).astype(np.float32)).unsqueeze(0).to('cuda:0')

        return last_ftState,last_imuState


    def Pref_callback(self,msgs):
        """接收来自仿真环境的位姿真值msg。维护一个字典{时间戳，位姿}用于保存最近接收的真值，用于对齐位姿估计值与真值的时间戳"""
        self.P_gt=np.array([
                msgs.transform.translation.x,
                msgs.transform.translation.y,
                msgs.transform.translation.z])
        self.P_gt_rot=np.array([
                msgs.transform.rotation.x,
                msgs.transform.rotation.y,
                msgs.transform.rotation.z,
                msgs.transform.rotation.w
            ])

        if self.cbf:
            self.P_ref_buf = {}#用于存放最近50ms的位置的真值
            self.cbf = False
        if self.P_ref_buf_recordFlag:
            timestamp = msgs.header.stamp.secs+msgs.header.stamp.nsecs*1e-9
            self.P_ref_buf[timestamp] = np.array([msgs.transform.translation.x,msgs.transform.translation.y,msgs.transform.translation.z])
            self.P_ref_ts = timestamp

    def Pest_callback(self,msgs):
        """接收来自slam系统的位姿估计值msg，每接收一个位姿估计值，就和位姿真值进行匹配，并且计算RPE"""
        self.P_est_ts = msgs.header.stamp.secs+msgs.header.stamp.nsecs*1e-9
        self.P_est = np.array([msgs.pose.position.x,msgs.pose.position.y,msgs.pose.position.z])
        self.P_est_rot = np.array([
            msgs.pose.orientation.x,
            msgs.pose.orientation.y,
            msgs.pose.orientation.z,
            msgs.pose.orientation.w
            ])
        self.getP_ref(self.P_est_ts)
        # record Align Traj
        if self.recordAlignTraj:
            self.alignTraj_est.append(self.P_est)
            # print(self.P_est)
            # print(f'current alignSet lenght:{len(self.alignTraj_est)}')
            self.alignTraj_ref.append(self.P_gt)
        # calculate the Rpe
        self.curRpe = self.getRpe()


    def getP_ref(self,timestamp):
        """输入位姿估计值的时间戳，找到对应的位姿真值，并且更新P_ref，更新后的P_ref与P_est的时间戳是对齐的"""
        self.P_ref_buf_recordFlag = False

        pose_gt = self.P_ref_buf.get(timestamp)
        if pose_gt is None:
            # 当P_est 没有找到对应的 P_ret时，寻找最近的P_ref
            pose_gt = list(self.P_ref_buf.items())[-1][1]
        self.P_ref = pose_gt
        self.cbf = True
        self.P_ref_buf_recordFlag = True

    # tenv.setTGU()
    def setTGU(self,OriginPoint):
        tUO = OriginPoint[0:3]
        tGO = self.P_gt
        qUO = OriginPoint[3:7]
        qGO = self.P_gt_rot

        RMUO = quaternion2RotationMatrix(qUO)
        RMGO = quaternion2RotationMatrix(qGO)

        TMUO = np.eye(4)
        TMUO[0:3,0:3] = RMUO
        TMUO[0:3,3] = tUO
        TMGO = np.eye(4)
        TMGO[0:3,0:3] = RMGO
        TMGO[0:3,3] = tGO

        TUO = se3(matrix=TMUO)
        TGO = se3(matrix=TMGO)
        self.TGU = TGO + TUO.exp().inv()

    def calcTU_Ustar(self):
        tUP = self.P_est
        tGP = self.P_gt
        qUP = self.P_est_rot
        qGP = self.P_gt_rot

        RMUP = quaternion2RotationMatrix(qUP)
        RMGP = quaternion2RotationMatrix(qGP)

        TMUP = np.eye(4)
        TMUP[0:3,0:3] = RMUP
        TMUP[0:3,3] = tUP
        TMGP = np.eye(4)
        TMGP[0:3,0:3] = RMGP
        TMGP[0:3,3] = tGP

        TUP = se3(matrix=TMUP)
        TGP = se3(matrix=TMGP)
        TGU_star = TGP + TUP.exp().inv()
        TU_Ustar = self.TGU.exp().inv() + TGU_star
        return TU_Ustar

    def calcDeltaTold(self):
        PUP = np.hstack((self.P_est,self.getEstYaw()))
        TGP = se3(vector=np.array([
            0,
            0,
            math.pi/180 * self.getCurYaw(),
            self.P_gt[0],
            self.P_gt[1],
            self.P_gt[2]
        ]))
        TUP = se3(vector=np.array([
            0,
            0,
            math.pi/180 * PUP[3],
            PUP[0],
            PUP[1],
            PUP[2]
        ]))
        CutTGU = TGP + TUP.exp().inv()
        # DeltaT = np.linalg.norm(CutTGU.w - self.TGU.w)
        DeltaT = CutTGU + self.TGU.exp().inv()
        return DeltaT

    def getCurYaw(self):
        x,y,z,w=self.P_gt_rot
        siny_cosp = 2* (z*w+x*y)
        cosy_cosp = 1 - 2 * (y*y + z*z)
        yaw = math.atan2(siny_cosp,cosy_cosp)/math.pi*180
        return yaw

    def getEstYaw(self):
        x,y,z,w=self.P_est_rot
        siny_cosp = 2* (z*w+x*y)
        cosy_cosp = 1 - 2 * (y*y + z*z)
        yaw = math.atan2(siny_cosp,cosy_cosp)/math.pi*180
        return yaw

    def getRpe(self):# change : 10hz
        # print('query RPE')
        """计算当前位姿估计值的最近3帧平均rpe"""
        if self.rpeWindow[self.rpeIdx] is None:
            self.rpeWindow[self.rpeIdx] = [self.P_est,self.P_ref]
            self.rpeIdx = (self.rpeIdx+1)%5
            return 0
        else:
            d_est_4_0 = self.rpeWindow[(self.rpeIdx+4)%5][0] - self.rpeWindow[self.rpeIdx][0]
            d_est_5_0 = self.P_est - self.rpeWindow[self.rpeIdx][0]

            d_ref_4_0 = self.rpeWindow[(self.rpeIdx+4)%5][1] - self.rpeWindow[self.rpeIdx][1]
            d_ref_5_0 = self.P_ref - self.rpeWindow[self.rpeIdx][1]

            self.rpeWindow[self.rpeIdx] = [self.P_est,self.P_ref]
            self.rpeIdx = (self.rpeIdx+1)%5

            return (np.linalg.norm(d_est_4_0-d_ref_4_0)+\
                    np.linalg.norm(d_est_5_0-d_ref_5_0))/2

    def connect(self):
        """连接ros系统，即创建状态订阅器，位姿真值订阅器，位姿估计订阅器。创建一个slam定位策略发布者。"""
        # State Subscriber
        self.stateSub = rospy.Subscriber("/Stereo_Inertial/ORB_SLAM3_msg", pixelstream, self.stateCallback,queue_size=1,buff_size=32768)
        # Estimate Pose Subscriber
        self.poseRefSub = rospy.Subscriber("hummingbird/ground_truth/transform", geometry_msgs.TransformStamped, self.Pref_callback)
        # Ground Pose Subscriber
        self.poseEstSub = rospy.Subscriber("Stereo_Inertial/ORB_SLAM3_EstPose", geometry_msgs.PoseStamped, self.Pest_callback)
        # ReAlignFlag Publisher
        self.ReAlignFlagPub = rospy.Publisher('/ReAlignFlag', std_msgs.Int8, queue_size=1)

    def disconnect(self):
        """注销订阅者与发布者"""
        self.stateSub.unregister()
        self.poseRefSub.unregister()
        self.poseEstSub.unregister()
        self.ReAlignFlagPub.unregister()

    def stepIpml(self,NextPoint):
        xt,yt,zt,wt,orient=NextPoint
        x0,y0,z0=tenv.P_est
        w0 = getEstYaw()
        vx=xt-x0
        vy=yt-y0
        vz=zt-z0
        vw=(wt-w0)*1/60 #这里有区别 负数
        tenv.env.uav.move_velo_time(vx,vy,vz,0,0,vw,0.5)

    def stepVoid(self): #空动作
        pass

    def func(self,x):
        return self.RE_a*(x-self.RE_b)

    def exp(self,x):
        return math.exp(x)-1

    def step(self,NextPoint):
        """无人机智在 t_step 的时间内前往下一目标点
        :param NextPoint : (x,y,z,w,orient)
        :return: Next_State-下一个状态, Reward-该动作的奖励, isDone-是否完成任务
        """
        ret_next_state = None
        expValid = False
        if not self.future.running():
            # print("执行第一个动作")
            self.future = self.threadPool.submit(self.stepIpml,NextPoint)
            time.sleep(0.48)
            # print("第一个动作不计算奖励")
            stepLog = {}
        else:
            # print("等待当前动作完成")
            while not self.future.done():
                time.sleep(0.0001)
            # print("当前动作完成，开始执行下一个动作")
            self.future = self.threadPool.submit(self.stepIpml,NextPoint)
            time.sleep(0.49)#或加一个判断：即将到达目标点附近

            self.curRe = self.getRewardE()# [-1,0]
            # print(f'curRe : {self.curRe}')
            self.curRg = self.getRewardG()# [-1,0]
            # print(f'curRg : {self.curRg}')
            # [-10,0]
            # self.reward = np.clip(self.WE * self.curRe + self.WG * self.curRg,self.R_min,self.R_max)
            self.reward = self.WE * self.curRe + self.WG * self.curRg
            stepLog = {
                't_gt':self.P_gt,
                'R_gt':self.P_gt_rot,
                't_est':self.P_est,
                'R_est':self.P_est_rot,
                'curRpe':self.curRpe,
                'Action':self.action,
                'curRe':self.curRe,
                'curRg':self.curRg,
                'TotalR':self.reward
            }
            expValid = True

        # 获取下一个状态
        next_ftState,next_imuState = self.getCurState(NextPoint[4])
        next_ftState = agent.StateNormalization(next_ftState)
        self.last_ftState = next_ftState
        self.last_imuState = next_imuState
        return next_ftState,next_imuState,self.reward,expValid,stepLog # return nextState, reward, done

    def stepOffline(self,NextPoint):
        ret_next_state = None
        expValid = False
        if not self.future.running():
            # print("执行第一个动作")
            self.future = self.threadPool.submit(self.stepIpml,NextPoint)
            time.sleep(0.48)
            # print("第一个动作不计算奖励")
            stepLog = {}
        else:
            # print("等待当前动作完成")
            while not self.future.done():
                time.sleep(0.0001)
            # print("当前动作完成，开始执行下一个动作")
            self.future = self.threadPool.submit(self.stepIpml,NextPoint)
            time.sleep(0.49)#或加一个判断：即将到达目标点附近
            # here
            self.curRe = self.getRewardE()# [-1,0]
            self.curRg = self.getRewardG()# [-1,0]
            self.reward = self.curRe + self.curRg
            # print(f'curRe: {self.curRe}, curRg: {self.curRg}, reward: {self.reward}')
            stepLog = {
                't_gt':self.P_gt,
                'R_gt':self.P_gt_rot,
                't_est':self.P_est,
                'R_est':self.P_est_rot,
                'curRpe':self.curRpe,
                'Action':self.action,
                'curRe':self.curRe,
                'curRg':self.curRg,
                'TotalR':self.reward
            }
            expValid = True

        # 获取下一个状态
        next_ftState,next_imuState = self.getCurStateOffline()
        self.last_ftState = next_ftState
        self.last_imuState = next_imuState
        return next_ftState,next_imuState,self.reward,expValid,stepLog # return nextState, reward, done

    # uav training control
    def makeSureSlamIsLaunchSuccessfully(self):
        while self.isSlamLaunchSucc is False:
            self.env.slamLauncher()
            # print("lanuching")
            # time.sleep(30)
            time.sleep(10)
            if self.checkSlamIsLaunching() is True:
                time.sleep(1)
                return
            else:
                env.slamShutdown()
                # print("Launching fail and retry")
                time.sleep(2)

    def checkSlamIsLaunching(self):
        IsError = False
        for i in range(5):
            if self.checkError():
                IsError = True
            time.sleep(0.1)
        if IsError:
            return False
        else:
            return  True

    def waitUavHover(self):
        while not self.env.uav.get_autopilot_state_name() is 'HOVER':
            time.sleep(0.01)

    def waitUavReadyForTask(self):
        while self.P_ref[2] < 6.9:
            time.sleep(0.01)
        time.sleep(2)

    def checkError(self):
        '''
        维护一个长度为5的窗口,用来记录slam的位姿估计,如果slam fail了,窗口内的位姿将不会更新
        如果窗口内的位姿都是一样的,说明slam fail了
        '''
        ret = True
        self.checkErrorBuffer[self.checkId%5]=self.P_est
        self.checkId+=1

        first = self.checkErrorBuffer[0][0]
        for i in self.checkErrorBuffer:
            if i[0] != first:
                ret = False
        return ret

    def OnlineCalibrPath(self,wd,DetectDeg):
        init_rate = rospy.Rate(1/2)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        # 前往任务起点
        self.env.uav.to_pose(0,0,4,0)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        self.waitUavHover()

        v = 0.4
        # set record flag True
        self.recordAlignTraj = True
        # online calibr
        self.env.uav.move_velo_time(-v,0,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(v,0,0,0,0,0,12)
        init_rate.sleep()
        self.env.uav.move_velo_time(-v,0,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,-v,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,v,0,0,0,0,12)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,-v,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,0,-v,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,0,v,0,0,0,12)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,0,-v,0,0,0,6)
        init_rate.sleep()
        # set record flag False
        self.recordAlignTraj = False
        self.env.uav.move_velo_time(0,v,0,0,0,0,3)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()

        theta0=(270+DetectDeg)/180*math.pi
        x = (20+wd)*math.cos(theta0)
        y = (20+wd)*math.sin(theta0)

        self.env.uav.to_pose(x,y,7,DetectDeg)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        self.waitUavHover()

    def OnlineCalibrPath_2(self,wd,DetectDeg):
        init_rate = rospy.Rate(1/2)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        # 前往任务起点
        self.env.uav.to_pose(0,0,4,0)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        self.waitUavHover()

        v = 0.4
        # set record flag True
        self.recordAlignTraj = True
        # online calibr
        self.env.uav.move_velo_time(-v,0,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(v,0,0,0,0,0,12)
        init_rate.sleep()
        self.env.uav.move_velo_time(-v,0,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,-v,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,v,0,0,0,0,12)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,-v,0,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,0,-v,0,0,0,6)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,0,v,0,0,0,12)
        init_rate.sleep()
        self.env.uav.move_velo_time(0,0,-v,0,0,0,6)
        init_rate.sleep()
        # set record flag False
        self.recordAlignTraj = False

        self.env.uav.move_velo_time(0,0,v,0,0,0,4)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()

        theta0=(270+DetectDeg)/180*math.pi
        x = (20+wd)*math.cos(theta0)
        y = (20+wd)*math.sin(theta0) + 3+20+wd

        self.env.uav.to_pose(x,y,7,DetectDeg)
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        init_rate.sleep()
        self.waitUavHover()

    def reset_alignTraj(self):
        self.alignTraj_ref = []
        self.alignTraj_est = []

In [ ]:
import torch
torch.cuda.is_available()

# 初始化任务轨迹
这个代码块定义了离线轨迹规划的代码

假设归一化平面的大小为2*3m
那么每次横向移动个割线为3m
圆周的角度为4.3deg

In [ ]:
# Task Path Initialized - Code block
def getCurYaw():
    x,y,z,w=tenv.P_gt_rot
    siny_cosp = 2* (z*w+x*y)
    cosy_cosp = 1 - 2 * (y*y + z*z)
    yaw = math.atan2(siny_cosp,cosy_cosp)/math.pi*180
    return yaw
def getEstYaw():
    x,y,z,w=tenv.P_est_rot
    siny_cosp = 2* (z*w+x*y)
    cosy_cosp = 1 - 2 * (y*y + z*z)
    yaw = math.atan2(siny_cosp,cosy_cosp)/math.pi*180
    return yaw

def moveToPoseAtOneSecond(goal):
    xt,yt,zt,wt,_,orient=goal
    x0,y0,z0=tenv.P_gt
    w0 = getCurYaw()
    vx=xt-x0
    vy=yt-y0
    vz=zt-z0
    vw=(wt-w0)*1/60
    tenv.env.uav.move_velo_time(vx,vy,vz,0,0,vw,1)

def SetSectorDeg(DetectionDeg:[],wallDist):#设置覆盖扇区的角度
    colDeg = []
    deg = 270#初始角度
    # ddeg = 4.3#设置横移步长，影响着巡检轨迹的列数
    R=20 + wallDist
    ddeg = 4.3
    for i in range(int(DetectionDeg[1]-DetectionDeg[0]/4.3)):
        theta = (deg+i*ddeg)/180*math.pi
        x = R*math.cos(theta)
        y = R*math.sin(theta)+ R
        colDeg.append(np.array([x,y,i*ddeg],dtype=float))
    return colDeg

def SetSectorDeg_2(DetectionDeg:[],wallDist):#设置覆盖扇区的角度
    colDeg = []
    deg = 270 + DetectionDeg[0] #初始角度

    # ddeg = 4.3#设置横移步长，影响着巡检轨迹的列数
    R=20 + wallDist
    ddeg = 4.3

    theta0=(deg)/180*math.pi
    xb = -R*math.cos(theta0)
    yb = -R*math.sin(theta0)

    for i in range(int((DetectionDeg[1]-DetectionDeg[0])/4.3)):
        theta = (deg+i*ddeg)/180*math.pi
        x = R*math.cos(theta)+ xb
        y = R*math.sin(theta)+ yb
        colDeg.append(np.array([x,y,i*ddeg],dtype=float))
    return colDeg

def SetSectorHeightRange(DetectingSector,high):#设置检查区域的高度
    TurningPointSet = []
    l=0
    h=high
    for i in range(0,len(DetectingSector)):
        x,y,w=DetectingSector[i]
        TurningPointSet.append((x,y,l,w,1,0))
        TurningPointSet.append((x,y,h,w,1,0))
        h,l = l,h#采用蛇形走位
    return TurningPointSet
def GenRoutePointSet(OriginPoint, TurningPointSet):#给定原点与轨迹，生成航线，并且设在每一个航迹点的方向
    RoutePointSet = []
    loop = [0, 1, 2, 1]
    x0, y0, z0, w0, _, _ = OriginPoint
    x, y, z, w, _, _ = TurningPointSet[0]
    RoutePointSet.append((x + x0, y + y0, z + z0, w + w0, 1, 0))
    for i in range(1, len(TurningPointSet)):
        x, y, z, w, _, _ = TurningPointSet[i]
        RoutePointSet.append((x + x0, y + y0, z + z0, w + w0, 1, loop[(i - 1) % 4]))
    return RoutePointSet

def CoveragePathPlanner(OriginPoint,DetectionDeg,high,wallDist):
    """
    新增: 调整与墙面的距离R
    巡检区域可视为圆柱体侧表面的某段弧形曲面，用左下角的起点，弧形的扇形大小，曲面的高度来进行定义.
    令U为无人机的坐标系原点
    slam的位置估计是在U坐标系下的
    因此航线的规划应该在U坐标系下直接规划。
    :param OriginPoint: 覆盖路径的起点
    :param DetectionDeg: 覆盖区域的扇形角度
    :param high: 覆盖区域的高度
    :param wallDist: 抵近测量距离
    :return: CoveragePointSet 巡检路径的拐点集
    :rtype: (x,y,z,w,orientation) orientation: 0 向上飞行 // 1 向下飞行 // 2 向右飞行
    """
    PUO = np.array(list(OriginPoint)[0:4])

    DetectingSector= SetSectorDeg(DetectionDeg,wallDist)
    x0,y0,z0,w0 = PUO[0:4]
    CoveragePointSet = []#PUP_Set
    l=0
    h=high
    for i in range(0,len(DetectingSector)):
        x,y,w=DetectingSector[i]
        CoveragePointSet.append([x0+x,y0+y,z0+l,w0+w,0])
        CoveragePointSet.append([x0+x,y0+y,z0+h,w0+w,0])
        h,l = l,h#采用蛇形走位

    loop = [0, 2, 1, 2]
    for i in range(1, len(CoveragePointSet)):
        CoveragePointSet[i][4] = loop[(i - 1) % 4]
    return CoveragePointSet

def CoveragePathPlanner_2(OriginPoint,DetectionDeg,high,wallDist):
    """
    新增: 调整与墙面的距离R
    巡检区域可视为圆柱体侧表面的某段弧形曲面，用左下角的起点，弧形的扇形大小，曲面的高度来进行定义.
    令U为无人机的坐标系原点
    slam的位置估计是在U坐标系下的
    因此航线的规划应该在U坐标系下直接规划。
    :param OriginPoint: 覆盖路径的起点
    :param DetectionDeg: 覆盖区域的扇形角度
    :param high: 覆盖区域的高度
    :param wallDist: 抵近测量距离
    :return: CoveragePointSet 巡检路径的拐点集
    :rtype: (x,y,z,w,orientation) orientation: 0 向上飞行 // 1 向下飞行 // 2 向右飞行
    """
    PUO = np.array(list(OriginPoint)[0:4])

    DetectingSector= SetSectorDeg_2(DetectionDeg,wallDist)
    x0,y0,z0,w0 = PUO[0:4]
    CoveragePointSet = []#PUP_Set
    l=0
    h=high
    for i in range(0,len(DetectingSector)):
        x,y,w=DetectingSector[i]
        CoveragePointSet.append([x0+x,y0+y,z0+l,w0+w,0])
        CoveragePointSet.append([x0+x,y0+y,z0+h,w0+w,0])
        h,l = l,h#采用蛇形走位

    loop = [0, 2, 1, 2]
    for i in range(1, len(CoveragePointSet)):
        CoveragePointSet[i][4] = loop[(i - 1) % 4]
    return CoveragePointSet

def DiscreteTraj(Traj,discreteDegree):#将航线离散化
    OutputTraj=[]
    n=len(Traj)
    for i in range(n-1):
        x0,y0,z0,w0,_,_=Traj[i]
        xt,yt,zt,wt,_,orient=Traj[i+1]
        norm = np.linalg.norm((xt-x0,yt-y0,zt-z0))
        N = math.floor(norm/discreteDegree)
        dx = (xt-x0)/N
        dy = (yt-y0)/N
        dz = (zt-z0)/N
        dw = (wt-w0)/N
        for j in range(N-1):
            OutputTraj.append((
                x0+(j+1)*dx,y0+(j+1)*dy,z0+(j+1)*dz,w0+(j+1)*dw,0,orient
            ))
        OutputTraj.append((
                x0+N*dx,y0+N*dy,z0+N*dz,w0+N*dw,1,orient
            ))
    return OutputTraj

def FollowByLOS(Traj,sight):#LOS算法轨迹跟踪
    n = len(Traj)
    x0,y0,z0 = tenv.P_gt#后期加入yaw
    for i in range(n):
        xt,yt,zt,_,isTp,_ = Traj[i]
        dist = np.linalg.norm((xt-x0,yt-y0,zt-z0))
        if isTp==1:#如果遍历到转角，飞往转角并悬停
            tenv.waitUavHover()
            moveToPoseAtOneSecond(Traj[i])
            tenv.waitUavHover()
            continue
        # 当前轨迹点不是转角，用LOS算法选择一个目标点
        if (dist<sight):
            continue
        else :
            # print(Traj[i-1])
            moveToPoseAtOneSecond(Traj[i-1])
#             x0,y0,z0 = tenv.P_gtsource devel/setup.bash
# source devel/setup.bash
            x0,y0,z0 = tenv.P_gtsource
            # source devel/setup.bash
import time
def setOriginTUMPoint():
    sum_x=0
    sum_y=0
    sum_z=0
    sum_q0=0
    sum_q1=0
    sum_q2=0
    sum_q3=0
    for i in range(20):
        x,y,z=tenv.P_est
        q0,q1,q2,q3=tenv.P_est_rot
        sum_x+=x
        sum_y+=y
        sum_z+=z
        sum_q0+=q0
        sum_q1+=q1
        sum_q2+=q2
        sum_q3+=q3
        time.sleep(0.05)
    return (sum_x/20,sum_y/20,sum_z/20,sum_q0/20,sum_q1/20,sum_q2/20,sum_q3/20,1,0)

def setOriginPoint():
    sum_x=0
    sum_y=0
    sum_z=0
    sum_w=0
    for i in range(20):
        x,y,z=tenv.P_est
        w=getEstYaw()
        sum_x+=x
        sum_y+=y
        sum_z+=z
        sum_w+=w
        time.sleep(0.05)
    return (sum_x/20,sum_y/20,sum_z/20,sum_w/20,1,0)

def GetTaskStartPoint():
    sum_x=0
    sum_y=0
    sum_z=0
    sum_w=0
    for i in range(20):
        x,y,z=tenv.P_est
        w=getEstYaw()
        sum_x+=x
        sum_y+=y
        sum_z+=z
        sum_w+=w
        time.sleep(0.05)
    return (sum_x/20,sum_y/20,sum_z/20,sum_w/20,0)

def moveToEstPoseAtOneSecond(goal):
    xt,yt,zt,wt,_,orient=goal
    x0,y0,z0=tenv.P_est
    w0 = getEstYaw()
    # print(f'{x0,y0,z0}')
    # print(f'w0:{w0},wt:{wt}')
    vx=xt-x0
    vy=yt-y0
    vz=zt-z0
    vw=(wt-w0)*1/60 #这里有区别 负数
    tenv.env.uav.move_velo_time(vx,vy,vz,0,0,vw,1)
    time.sleep(0.1)

def FollowingTrajBySlam(Traj,sight):#LOS算法轨迹跟踪
    n = len(Traj)
    x0,y0,z0 = tenv.P_est
    for i in range(n):
        xt,yt,zt,_,isTp,_ = Traj[i]
        dist = np.linalg.norm((xt-x0,yt-y0,zt-z0))
        if isTp==1:#如果遍历到转角，飞往转角并悬停
            print(f'arrived TP:{Traj[i]}')
            print(f'dt = {np.linalg.norm(tenv.P_est - tenv.P_gt)}')
            tenv.waitUavHover()
            moveToEstPoseAtOneSecond(Traj[i])
            tenv.waitUavHover()
            continue
        # 当前轨迹点不是转角，用LOS算法选择一个目标点
        if (dist<sight):
            continue
        else :
            moveToEstPoseAtOneSecond(Traj[i-1])
            x0,y0,z0 = tenv.P_est
def printP_est():
    print(f'X = {tenv.P_est[0]}')
    print(f'Y = {tenv.P_est[1]}')
    print(f'Z = {tenv.P_est[2]}')
    print(f'Yaw = {getEstYaw()}')
def printP_gt():
    print(f'X = {tenv.P_gt[0]}')
    print(f'Y = {tenv.P_gt[1]}')
    print(f'Z = {tenv.P_gt[2]}')
    print(f'Yaw = {getCurYaw()}')

def moveToEstPoseAtNSecond(goal,N):
    xt,yt,zt,wt,_,orient=goal
    x0,y0,z0=tenv.P_est
    w0 = getEstYaw()
    vx=(xt-x0)/N
    vy=(yt-y0)/N
    vz=(zt-z0)/N
    vw=((wt-w0)*(1/60))/N #这里有区别 负数
    tenv.env.uav.move_velo_time(vx,vy,vz,0,0,vw,N)
    time.sleep(0.1)

# 巡检路径跟踪算法
这个代码块定义了控制无人机对给定离线轨迹进行跟踪的算法

In [ ]:
# Path Following algorithm - Code block
def lineseg_dist(P, A, B):# 计算 P点 到  A B 所确定的直线的最短距离
    return np.linalg.norm(np.cross(B-A,P-A))/np.linalg.norm(B-A)

def CalcNextPoint(CurPosition,PrevTargetPoint,CurTargetPoint,ViewDist):
    ## 计算下一个目标点
    # 变量见实验笔记
    # CurPosition:C PrevPosition:A  TarPosition:B
    PrevPosition = np.array(list(PrevTargetPoint[:3]))
    TarPosition = np.array(list(CurTargetPoint[:3]))
    orient = int((PrevPosition[2] - TarPosition[2])>0) # 向上飞行：0 向下飞行：1
    Za = PrevPosition[2]
    Zc = CurPosition[2]

    v = ViewDist
    a = np.linalg.norm(CurPosition - PrevPosition)
    d = lineseg_dist(CurPosition,PrevPosition,TarPosition)

    tmp = a*a - d*d
    if tmp<0:
        tmp = 0
    b = math.sqrt(tmp)

    tmp = v*v - d*d
    if tmp<0:
        tmp = 0
    c = math.sqrt(tmp)
    if orient == 0:# 向上飞行
        if Za < Zc:# C 在 A B 之间
            NextPoint = np.array([PrevTargetPoint[0],PrevTargetPoint[1],PrevTargetPoint[2]+b+c,PrevTargetPoint[3],orient])
        else:# C 在 A B 之外
            NextPoint = np.array([PrevTargetPoint[0],PrevTargetPoint[1],PrevTargetPoint[2]+c-b,PrevTargetPoint[3],orient])
    else: # 向下飞行
        if Za > Zc:# C 在 A B 之间
            NextPoint = np.array([PrevTargetPoint[0],PrevTargetPoint[1],PrevTargetPoint[2]-b-c,PrevTargetPoint[3],orient])
        else:# C 在 A B 之外
            NextPoint = np.array([PrevTargetPoint[0],PrevTargetPoint[1],PrevTargetPoint[2]-c+b,PrevTargetPoint[3],orient])
    return NextPoint

def CarefullyMoveToPoint(point,t):
    xt,yt,zt,wt,orient=point
    x0,y0,z0=tenv.P_est
    w0 = getEstYaw()
    vx=(xt-x0)/t
    vy=(yt-y0)/t
    vz=(zt-z0)/t
    vw=((wt-w0)*1/60)/t
    tenv.env.uav.move_velo_time(vx,vy,vz,0,0,vw,t)

def moveDeltaPoseAtTimeT(x,y,z,h,t):
    vx=x/t
    vy=y/t
    vz=z/t
    yawh=h*1/60/t
    tenv.env.uav.move_velo_time(vx,vy,vz,0,0,yawh,t)

def CarefullyFlyAround(A,B,O):
    # 给定两个点，生成这两个点对应在园上的弧的 离散的点
    initDeg = 270 + A[3] - O[3]
    initPos = O
    ddeg = 0.5
    pointNum = int((B[3]-A[3])/0.5)
    R=21
    arc = []
    for i in range(1,pointNum):
        theta = (initDeg+i*ddeg)/180*math.pi
        x = R*math.cos(theta)
        y = R*math.sin(theta)+21
        arc.append(np.array([x,y,0],dtype=float)) # (0,0,10)

    for i in range(len(arc)):
        arc[i][0] = arc[i][0] + initPos[0]
        arc[i][1] = arc[i][1] + initPos[1]

    preX = A[0]
    preY = A[1]
    for p in arc:
        # 计算 delta x delta y
        dx = p[0]-preX
        dy = p[1]-preY
        preX = p[0]
        preY = p[1]
        moveDeltaPoseAtTimeT(dx,dy,0,0.575,1)

def CarefullyFlyAround_2(DetectDeg,A,B,O):
    # 给定两个点，生成这两个点对应在园上的弧的 离散的点
    initDeg = 270 + DetectDeg[0] + A[3] - O[3]
    initPos = A
    ddeg = 0.5
    pointNum = int((B[3]-A[3])/0.5)
    R=21
    arc = []

    theta0=(initDeg)/180*math.pi
    xb = -R*math.cos(theta0)
    yb = -R*math.sin(theta0)
    # 以
    for i in range(1,pointNum):
        theta = (initDeg+i*ddeg)/180*math.pi
        x = R*math.cos(theta)+ xb
        y = R*math.sin(theta)+ yb

        arc.append(np.array([x,y,0],dtype=float)) # (0,0,10)

    for i in range(len(arc)):
        arc[i][0] = arc[i][0] + initPos[0]
        arc[i][1] = arc[i][1] + initPos[1]

    preX = A[0]
    preY = A[1]
    for p in arc:
        # 计算 delta x delta y
        dx = p[0]-preX
        dy = p[1]-preY
        preX = p[0]
        preY = p[1]
        moveDeltaPoseAtTimeT(dx,dy,0,0.575,1)

def CarefullyTurnAround(CoveragePointSet,CurTargetIdx):
    O = CoveragePointSet[0]
    A = CoveragePointSet[CurTargetIdx]
    B = CoveragePointSet[CurTargetIdx+1]
    tenv.waitUavHover()
    CarefullyMoveToPoint(A,2)
    tenv.waitUavHover()
    CarefullyFlyAround(A,B,O)
    tenv.waitUavHover()
    CarefullyMoveToPoint(B,2)
    tenv.waitUavHover()

def CarefullyTurnAround_2(DetectDeg,CoveragePointSet,CurTargetIdx):
    O = CoveragePointSet[0]
    A = CoveragePointSet[CurTargetIdx]
    B = CoveragePointSet[CurTargetIdx+1]
    tenv.waitUavHover()
    CarefullyMoveToPoint(A,2)
    tenv.waitUavHover()
    CarefullyFlyAround_2(DetectDeg,A,B,O)
    tenv.waitUavHover()
    CarefullyMoveToPoint(B,2)
    tenv.waitUavHover()

def checkIsDeviated(CoveragePointSet,PrevTargetIdx,CurTargetIdx):
    if PrevTargetIdx>=len(CoveragePointSet):
        return False
    PrevTargetPoint=CoveragePointSet[PrevTargetIdx]
    CurTargetPoint=CoveragePointSet[CurTargetIdx]
    PrevPosition = np.array(list(PrevTargetPoint[:3]))
    TarPosition = np.array(list(CurTargetPoint[:3]))
    d = lineseg_dist(tenv.P_est,PrevPosition,TarPosition)
    print(f'与GT航线的最短距离为{d}')
    return d>0.5

def calcDeviatedDist(CoveragePointSet,PrevTargetIdx,CurTargetIdx):
    if PrevTargetIdx>=len(CoveragePointSet):
        return False
    PrevTargetPoint=CoveragePointSet[PrevTargetIdx]
    CurTargetPoint=CoveragePointSet[CurTargetIdx]
    PrevPosition = np.array(list(PrevTargetPoint[:3]))
    TarPosition = np.array(list(CurTargetPoint[:3]))
    d = lineseg_dist(tenv.P_est,PrevPosition,TarPosition)
    # print(f'与GT航线的最短距离为{d}')
    return d

# 在线校正
import numpy as np
import typing

UmeyamaResult = typing.Tuple[np.ndarray, np.ndarray, float]

def umeyama_alignment(x: np.ndarray, y: np.ndarray, with_scale: bool = False) -> UmeyamaResult:
    """
    Computes the least squares solution parameters of an Sim(m) matrix
    that minimizes the distance between a set of registered points.
    Umeyama, Shinji: Least-squares estimation of transformation parameters
                     between two point patterns. IEEE PAMI, 1991
    :param x: mxn matrix of points, m = dimension, n = nr. of data points
    :param y: mxn matrix of points, m = dimension, n = nr. of data points
    :param with_scale: set to True to align also the scale (default: 1.0 scale)
    :return: r, t, c - rotation matrix, translation vector and scale factor
    """
    # m = dimension, n = nr. of data points
    m, n = x.shape

    # means, eq. 34 and 35
    mean_x = x.mean(axis=1)
    mean_y = y.mean(axis=1)

    # variance, eq. 36
    # "transpose" for column subtraction
    sigma_x = 1.0 / n * (np.linalg.norm(x - mean_x[:, np.newaxis])**2)

    # covariance matrix, eq. 38
    outer_sum = np.zeros((m, m))
    for i in range(n):
        outer_sum += np.outer((y[:, i] - mean_y), (x[:, i] - mean_x))
    cov_xy = np.multiply(1.0 / n, outer_sum)

    # SVD (text betw. eq. 38 and 39)
    u, d, v = np.linalg.svd(cov_xy)
    if np.count_nonzero(d > np.finfo(d.dtype).eps) < m - 1:
        print("Degenerate covariance rank, Umeyama alignment is not possible")

    # S matrix, eq. 43
    s = np.eye(m)
    if np.linalg.det(u) * np.linalg.det(v) < 0.0:
        # Ensure a RHS coordinate system (Kabsch algorithm).
        s[m - 1, m - 1] = -1

    # rotation, eq. 40
    r = u.dot(s).dot(v)

    # scale & translation, eq. 42 and 41
    c = 1 / sigma_x * np.trace(np.diag(d).dot(s)) if with_scale else 1.0
    t = mean_y - np.multiply(c, r.dot(mean_x))

    return r, t, c
def writeOnlineAlignTxt():
    x = tenv.alignTraj_est
    y = tenv.alignTraj_ref
    x = np.asarray(x).transpose()
    y = np.asarray(y).transpose()
    R,t,s = umeyama_alignment(x,y)
    with open("/home/dbq/dbq/DRL_SLAM/slam/exp/fm_orbslam3/stereoInertial/OnlineAlign.txt",'w') as f:
        for r in R:
            for c in r:
                f.write(f'{float(c)}\n')
        for r in t:
            f.write(f'{float(r)}\n')

def calcAlignT():
    x = tenv.alignTraj_est
    y = tenv.alignTraj_ref
    x = np.asarray(x).transpose()
    y = np.asarray(y).transpose()
    R,t,s = umeyama_alignment(x,y)
    return R,t

def ReAlignSLAM():
    intmsg = std_msgs.Int8()
    intmsg.data= 0
    tenv.ReAlignFlagPub.publish(intmsg)

In [ ]:
# PPO Agent Code Block
# 这一个代码块定义了 Agent 的网络结构、训练超参数、网络更新函数等功能
import os
import time
import numpy as np
import math
import torch
import torch.nn as nn
from torch import Tensor
from torch.distributions.normal import Normal
from torch.nn import functional as F

def build_mlp(dims: [int]) -> nn.Sequential:  # MLP (MultiLayer Perceptron)
    net_list = []
    for i in range(len(dims) - 1):
        net_list.extend([nn.Linear(dims[i], dims[i + 1]), nn.Tanh()])# Trick 10 : Tanh activation func
    del net_list[-1]  # remove the activation of output layer
    return nn.Sequential(*net_list)

class ActorCnn(nn.Module):
    def __init__(self, dims: [int], state_dim: int, action_dim: int,action_std_log_init:float):
        super().__init__()
        # 动作方差的对数
        self.action_std_log = nn.Parameter(torch.tensor(action_std_log_init), requires_grad=True)

        self.conv1 = nn.Sequential(         # input shape (1, 48, 72)
            nn.Conv2d(1,16,8,4,2),
            nn.ReLU(),                      # output shape (8, 12, 18)
            nn.Conv2d(16,32,4,2,0),
            nn.ReLU(),                      # output shape (32, 5, 8)
            nn.MaxPool2d(2),                # output shape (32, 2, 4)
        )

        self.conv2 = nn.Sequential(         # input shape (32, 2, 4)
            nn.Conv2d(32, 64, 2, 1, 0),     # output shape (64, 1, 3)
            nn.ReLU(),                      # activation
        )

        self.fc1 = nn.Linear(64 * 1 * 3, state_dim - 6)   #  fully connected layer 1

        self.fc2 = build_mlp(dims=[state_dim, *dims, action_dim]) #  fully connected layer 1

    def forward(self, ftState: Tensor, imuState: Tensor) -> Tensor: # [-1, 1]
        x = self.conv1(ftState)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)# flatten ft map
        x = F.tanh(self.fc1(x))# fully connected layer 1
        x = torch.cat((x,imuState),dim = -1)# cat dynamic info
        x = F.tanh(self.fc2(x))# fully connected layer 2
        return x

    # 获取动作与动作的概论值
    def get_action(self, ftState: Tensor, imuState: Tensor) -> (Tensor, Tensor):  # for exploration
        # 获得动作的分布
        action_avg = self.forward(ftState,imuState) # S -> [-1,1]
        action_std = self.action_std_log.exp()
        # 采样
        dist = Normal(action_avg, action_std)#取决于std，如果std取值不当，会使值映射在[-1,1]区间外的范围，如std=0,[-4,4]
        action = dist.sample()# 采样动作
        logprob = dist.log_prob(action)# 计算采样到该动作的概论
        return action, logprob

    def get_logprob_entropy(self, ftState: Tensor, imuState: Tensor, action: Tensor) -> (Tensor, Tensor):
        action_avg = self.forward(ftState,imuState)
        action_std = self.action_std_log.exp()

        dist = Normal(action_avg, action_std)
        logprob = dist.log_prob(action)
        entropy = dist.entropy()
        return logprob, entropy

class CriticCnn(nn.Module):
    def __init__(self, dims: [int], state_dim: int, _action_dim: int):
        super().__init__()
        self.conv1 = nn.Sequential(         # input shape (1, 48, 72)
            nn.Conv2d(1,16,8,4,2),
            nn.ReLU(),                      # output shape (8, 12, 18)
            nn.Conv2d(16,32,4,2,0),
            nn.ReLU(),                      # output shape (32, 5, 8)
            nn.MaxPool2d(2),                # output shape (32, 2, 4)
        )
        self.conv2 = nn.Sequential(         # input shape (32, 2, 4)
            nn.Conv2d(32, 64, 2, 1, 0),     # output shape (64, 1, 3)
            nn.ReLU(),                      # activation
        )
        self.fc1 = nn.Linear(64 * 1 * 3, state_dim - 6)   #  fully connected layer
        self.fc2 = build_mlp(dims=[state_dim, *dims, 1])

    def forward(self, ftState: Tensor, imuState: Tensor) -> Tensor:
        x = self.conv1(ftState)                 # conv 2d layer 1
        x = self.conv2(x)                       # conv 2d layer 2
        x = x.view(x.size(0), -1)               # flatten ft map
        x = self.fc1(x).tanh()              # fully connected layer 1
        x = torch.cat((x,imuState),dim = -1)    # cat dynamic info
        x = self.fc2(x)                         # fully connected layer 2
        return x

class ActorFc(nn.Module):
    def __init__(self, dims: [int], state_dim: int, action_dim: int,action_std_log_init:float):
        super().__init__()
        # 动作方差的对数
        self.action_std_log = torch.tensor(action_std_log_init).to('cuda:0')

        # 创建全链接网络
        self.net = build_mlp(dims=[state_dim, *dims, action_dim])

    def forward(self, ftState: Tensor, imuState: Tensor) -> Tensor:
        x = torch.cat((ftState,imuState),dim = -1)# cat dynamic info
        return self.net(x).tanh()  # action.tanh()

    # 获取动作与动作的概论值
    def get_action(self, ftState: Tensor, imuState: Tensor) -> Tensor:  # for exploration
        # 获得动作的分布
        action_avg = self.forward(ftState,imuState)
        action_std = self.action_std_log.exp()
        # 采样
        dist = Normal(action_avg, action_std)
        action = dist.sample()# 采样动作
        logprob = dist.log_prob(action)# 计算采样到该动作的概论
        return action, logprob, action_avg

    def get_logprob_entropy(self, ftState: Tensor, imuState: Tensor, action: Tensor) -> (Tensor, Tensor):
        action_avg = self.forward(ftState,imuState)
        action_std = self.action_std_log.exp()

        dist = Normal(action_avg, action_std)
        logprob = dist.log_prob(action)
        entropy = dist.entropy()
        return logprob, entropy

    def set_action_std(self, new_action_std):
            self.action_std_log = torch.tensor(new_action_std).to('cuda:0')

    @staticmethod
    def convert_action_for_env(action: Tensor) -> Tensor:
        return action.tanh()

class CriticFc(nn.Module):
    def __init__(self, dims: [int], state_dim: int, _action_dim: int):
        super().__init__()
        self.net = build_mlp(dims=[state_dim, *dims, 1])

    def forward(self, ftState: Tensor, imuState: Tensor) -> Tensor:
        x = torch.cat((ftState,imuState),dim = -1)# cat dynamic info
        return self.net(x)

class PPO_Config:
    def __init__(self):
        # 网络结构
        self.NetArc = 'fc' # fc cnn
        if self.NetArc == 'fc':
            self.state_dim = 400
            # self.state_dim = 384 # ctx
            self.net_dims = [128,128,64]
        else:
            self.state_dim = 128
            self.net_dims = [128,64]

        self.action_dim = 1  # vector dimension (feature number) of action

        self.action_std_log = -1.5
        self.action_std_decay_rate = 0.1
        self.min_action_std = -4

        self.sycnUd = False # ctx

        self.gpu_id = 0
        self.device = torch.device(f"cuda:{self.gpu_id}" if (torch.cuda.is_available() and (self.gpu_id >= 0)) else "cpu")

        self.gamma = 0.90  # 折扣率 设置为0.95的理由是 失败一般是由短期的速度不当引起的
        self.reward_scale = 1.0  # an approximate target reward usually be closed to 256
        self.ratio_clip = 0.25  # PPO2中优势函数的截断值 `ratio.clamp(1 - clip, 1 + clip)`
        self.lambda_gae_adv = 0.98  # GAE的lambda值 could be 0.80~0.99
        self.lambda_entropy = 0.01  #  could be 0.00~0.10
        self.lambda_entropy = torch.tensor(self.lambda_entropy, dtype=torch.float32, device=self.device)

        '''Arguments for training'''
        self.batch_size = int(512)  # num of transitions sampled from replay buffer.
        self.horizon_len = int(10)  # collect horizon_len step while exploring, then update network ctx: 1024
        self.repeat_times = 16.0 # repeatedly update network using ReplayBuffer to keep critic's loss small

        '''Arguments for device'''
        self.gpu_id = int(0)  # `int` means the ID of single GPU, -1 means CPU
        # Trick 6 : Learning Rate Decay
        self.usedLRD = False
        self.update_target_time = 120
        self.max_total_steps = self.update_target_time * self.horizon_len * self.repeat_times
        self.learning_rate = 6e-5  # 学习率  2 ** -14 ~= 6e-5
        # Phi net path
        self.PhiNetPath = 'no path'


class RunningMeanStd:
    # Dynamically calculate mean and std
    def __init__(self, shape):  # shape:the dimension of input data
        self.n = 0
        self.mean = 0.0
        self.S = 0.0
        self.std = math.sqrt(self.S)
        self.shape = shape

    def update(self, x):
        self.n += 1
        if self.n == 1:
            self.mean = x.mean()
            self.std = x.std()
        else:
            x_CurMean = x.mean()
            old_mean = self.mean
            self.mean = old_mean + (x_CurMean - old_mean) / (self.n * self.shape)
            self.S = self.S + (x_CurMean - old_mean) * (x_CurMean - self.mean)
            self.std = math.sqrt(self.S / self.n )

class Normalization:
    def __init__(self,r,c):
        self.running_ms = RunningMeanStd(shape=r*c)
        self.shape = r*c
        self.r = r
        self.c = c
        self.update = True
    def __call__(self, x):
        # Whether to update the mean and std,during the evaluating,update=Flase
        if self.update:
            self.running_ms.update(x)
        mu = torch.ones((self.r,self.c)).to('cuda:0') * self.running_ms.mean
        std = torch.ones((self.r,self.c)).to('cuda:0') * self.running_ms.std
        x = (x-mu)/std
        return x

class AgentPPO():
    def __init__(self, cfg: PPO_Config = PPO_Config()):
        '''获取参数'''
        self.net_dims = cfg.net_dims
        self.state_dim = cfg.state_dim
        self.action_dim = cfg.action_dim

        self.action_std_log = cfg.action_std_log
        self.action_std_decay_rate = cfg.action_std_decay_rate
        self.min_action_std = cfg.min_action_std

        self.gamma = cfg.gamma
        self.reward_scale = cfg.reward_scale
        self.ratio_clip = cfg.ratio_clip
        self.lambda_gae_adv = cfg.lambda_gae_adv
        self.lambda_entropy =  cfg.lambda_entropy

        self.learning_rate = cfg.learning_rate
        self.horizon_len = cfg.horizon_len
        self.repeat_times = cfg.repeat_times
        self.batch_size = cfg.batch_size

        self.NetArc = cfg.NetArc
        # self.PhiNetPath = cfg.PhiNetPath

        self.last_state = None  # save the last state of the trajectory for training. `last_state.shape == (state_dim)`
        self.device = torch.device(f"cuda:{cfg.gpu_id}" if (torch.cuda.is_available() and (cfg.gpu_id >= 0)) else "cpu")

        # 定义actor与critic
        if self.NetArc == 'fc':
            self.act = ActorFc(self.net_dims, self.state_dim, self.action_dim,self.action_std_log).to(self.device)
            self.cri = CriticFc(self.net_dims, self.state_dim, self.action_dim).to(self.device)
        # else:
        #     self.act = ActorCnn(self.net_dims, self.state_dim, self.action_dim,self.action_std_log).to(self.device)
        #     self.cri = CriticCnn(self.net_dims, self.state_dim, self.action_dim).to(self.device)

        # 定义优化器 # Trick 9 : Adam Optimizer Epsillon Parameter
        self.act_optimizer = torch.optim.Adam(self.act.parameters(), self.learning_rate,eps=1e-5)
        self.cri_optimizer = torch.optim.Adam(self.cri.parameters(), self.learning_rate,eps=1e-5)
        # 定义两个网络参数距离的标准
        self.criterion = torch.nn.SmoothL1Loss()
        # Trick 2 : State Normaliztion
        if self.NetArc == 'fc':
            self.StateNormalization = Normalization(r=1,c=384)# only normalize vision feature
        else:
            self.StateNormalization = Normalization(r=48,c=72)# only normalize vision feature

        # Trick 6 : Learning Rate Decay
        self.usedLRD = cfg.usedLRD
        self.total_steps = 0
        self.update_target_time = cfg.update_target_time
        self.max_total_steps = cfg.max_total_steps

        # 是否同步更新actor和critic
        self.sycnUd = cfg.sycnUd

        # # 定义Phi
        # self.Phi = CriticPPO(self.net_dims, self.state_dim, self.action_dim).to(self.device)
        # self.Phi.load_state_dict(torch.load(self.PhiNetPath, map_location=lambda storage, loc: storage))

        # critic loss
        self.criticLoss = []

    def lr_decay(self,total_steps):
        lr_now = self.learning_rate * (1 - total_steps/self.max_total_steps)
        self.act_optimizer.param_groups[0]['lr'] = lr_now
        self.cri_optimizer.param_groups[0]['lr'] = lr_now

    def set_action_std(self, new_action_std):
        self.action_std_log = new_action_std
        self.act.set_action_std(new_action_std)

    def decay_action_std(self):
        self.action_std_log = self.action_std_log - self.action_std_decay_rate
        self.action_std_log = round(self.action_std_log, 4)
        if (self.action_std_log <= self.min_action_std):
            self.action_std_log = self.min_action_std
        self.set_action_std(self.action_std_log)

    def update_net(self, buffer) -> [float]:
        # 计算Value，该Value是用behavior critic 算出来的
        with torch.no_grad():
            ftStates, imuStates, actions, logprobs, rewards, undones = buffer
            buffer_size = ftStates.shape[0]
            '''get advantages reward_sums'''
            bs = 2 ** 10  # set a smaller 'batch_size' when out of GPU memory.
            values = [self.cri(ftStates[i:i + bs],imuStates[i:i + bs]) for i in range(0, buffer_size, bs)]
            values = torch.cat(values, dim=0).squeeze(1)

            advantages = self.get_advantages(rewards, undones, values)  # 计算GAE优势函数值

            reward_sums = advantages + values  # reward_sums = reward + next_value = Q
            del rewards, undones, values

            # Trick 1: Advantage Normalization ：： 使用GAE计算完一个batch中的advantage后，计算整个batch中所有advantage的mean和std，然后减均值再除以标准差。
            advantages = (advantages - advantages.mean()) / (advantages.std(dim=0) + 1e-5)
        assert logprobs.shape == advantages.shape == reward_sums.shape == (buffer_size,)

        '''update network'''
        obj_critics = 0.0
        obj_actors = 0.0

        update_times = int(buffer_size * self.repeat_times / self.batch_size)# 使用这些经验更新很多次
        assert update_times >= 1
        for repId in range(update_times):
            print(f'repId:{repId}')
            # 随机获取一个batch的经验
            indices = torch.randint(buffer_size, size=(self.batch_size,), requires_grad=False)
            ftState = ftStates[indices]
            imuState = imuStates[indices]
            action = actions[indices]
            logprob = logprobs[indices]
            advantage = advantages[indices]
            reward_sum = reward_sums[indices]

            # 更新Learning Rate
            if self.usedLRD:
                self.lr_decay(self.total_steps)

            # 先更新 critic
            value = self.cri(ftState,imuState).squeeze(1)  # V值 ，reward_sum 为 Q 值
            obj_critic = self.criterion(value, reward_sum)# 用smoothL1计算 Q - V ，即TD-error
            self.optimizer_update(self.cri_optimizer, obj_critic)# ctitic的更新用TD-error作为loss，自举

            # # 每一轮只更新4次actor
            # if self.sycnUd or repId < 3 :

            # 这里的act是更新后的 behavior policy, state 和 action 是原来的behavior policy收集的
            new_logprob, obj_entropy = self.act.get_logprob_entropy(ftState, imuState, action)# 输出对数概率值 策略的熵
            ratio = (new_logprob - logprob.detach()).exp()# 重要性采样：更新后的分布的采样概率/行为分布的采样概率
            surrogate1 = advantage * ratio
            surrogate2 = advantage * ratio.clamp(1 - self.ratio_clip, 1 + self.ratio_clip)
            obj_surrogate = torch.min(surrogate1, surrogate2).mean()# PPO2 clip

            obj_actor = obj_surrogate + obj_entropy.mean() * self.lambda_entropy# J + KL
            self.optimizer_update(self.act_optimizer, -obj_actor)# 用PG的方式更新actor

            obj_critics += obj_critic.item()# 统计loss
            obj_actors += obj_actor.item()# 统计loss

            # 总更新步数增加一个batch size
            self.total_steps += self.batch_size

        a_std_log = self.act.action_std_log.mean()
        return obj_critics / update_times, obj_actors / update_times, a_std_log.item()

    def update_critic(self, buffer) -> float:
        # 计算Value，该Value是用 behavior critic 算出来的
        with torch.no_grad():
            ftStates, imuStates, actions, logprobs, rewards, undones = buffer
            buffer_size = ftStates.shape[0]
            '''get advantages reward_sums'''
            bs = 2 ** 10  # set a smaller 'batch_size' when out of GPU memory.
            values = [self.cri(ftStates[i:i + bs],imuStates[i:i + bs]) for i in range(0, buffer_size, bs)]
            values = torch.cat(values, dim=0).squeeze(1)

            advantages = self.get_advantages(rewards, undones, values)  # 计算GAE优势函数值

            reward_sums = advantages + values  # reward_sums = reward + next_value = Q
            del rewards, undones, values

            # Trick 1: Advantage Normalization ：： 使用GAE计算完一个batch中的advantage后，计算整个batch中所有advantage的mean和std，然后减均值再除以标准差。
            advantages = (advantages - advantages.mean()) / (advantages.std(dim=0) + 1e-5)
        assert logprobs.shape == advantages.shape == reward_sums.shape == (buffer_size,)

        '''update network'''
        obj_critics = 0.0

        update_times = int(buffer_size * self.repeat_times / self.batch_size)# 使用这些经验更新很多次
        assert update_times >= 1
        for udCnt in range(update_times):
            # 随机获取一个batch的经验
            indices = torch.randint(buffer_size, size=(self.batch_size,), requires_grad=False)
            ftState = ftStates[indices]
            imuState = imuStates[indices]
            action = actions[indices]
            logprob = logprobs[indices]
            advantage = advantages[indices]
            reward_sum = reward_sums[indices]

            # 更新Learning Rate
            if self.usedLRD:
                self.lr_decay(self.total_steps)
                print(self.cri_optimizer.param_groups[0]['lr'])

            # 更新 critic
            value = self.cri(ftState,imuState).squeeze(1)  # V值 ，reward_sum 为 Q 值
            obj_critic = self.criterion(value, reward_sum) # 用smoothL1计算 Q - V ，即TD-error
            self.optimizer_update(self.cri_optimizer, obj_critic) # ctitic的更新用TD-error作为loss，自举

            obj_critics += obj_critic.item() # 统计loss
            # print(obj_critic.item())
            self.criticLoss.append(obj_critic.item())

            # 总更新步数增加一个batch size
            self.total_steps += self.batch_size
            b = 10
            if udCnt % b == 0:
                # 保存critic
                save_path = SavePath+train_name+"/checkpoints/"+f'critic_{int(udCnt / b)}.pth'
                torch.save(agent.cri.state_dict(), save_path)
        return obj_critics / update_times

    def get_advantages(self, rewards: Tensor, undones: Tensor, values: Tensor) -> Tensor:
        advantages = torch.empty_like(values)  # advantage value

        masks = undones * self.gamma
        horizon_len = rewards.shape[0]

        next_value = 0

        advantage = 0  # last_gae_lambda
        for t in range(horizon_len - 1, -1, -1):
            delta = rewards[t] + masks[t] * next_value - values[t]
            advantages[t] = advantage = delta + masks[t] * self.lambda_gae_adv * advantage
            next_value = values[t]
        return advantages

    @staticmethod
    def optimizer_update(optimizer, objective: Tensor):
        optimizer.zero_grad()
        objective.backward()
        # # Trick 7 : Gradient clip
        # nn.utils.clip_grad_norm(objective,0.5)
        optimizer.step()

# 模型训练
下面的代码块用于训练DRL模型

In [ ]:
# tenv.disconnect()
# env.slamShutdown()

In [ ]:
# PPO Agent Config - Code block
class PPO_Config:
    def __init__(self):
        # 网络结构
        self.NetArc = 'fc' # fc cnn

        if self.NetArc == 'fc':
            self.state_dim = 390
            self.net_dims = [390,256,128,64]
        else:
            self.state_dim = 128
            self.net_dims = [128,64]

        self.action_dim = 1  # vector dimension (feature number) of action

        self.gpu_id = 0
        self.device = torch.device(f"cuda:{self.gpu_id}" if (torch.cuda.is_available() and (self.gpu_id >= 0)) else "cpu")

        self.action_std_log = -0.7 # -1.0
        self.action_std_decay_rate = 0.05 # 3/0.05 * 2 = 120
        self.min_action_std = -4.0

        self.gamma = 0.95  # 折扣率 设置为0.95的理由是 失败一般是由短期的速度不当引起的
        self.reward_scale = 1.0  # an approximate target reward usually be closed to 256
        self.ratio_clip = 0.25  # PPO2中优势函数的截断值 `ratio.clamp(1 - clip, 1 + clip)`
        self.lambda_gae_adv = 0.98  # GAE的lambda值 could be 0.80~0.99
        self.lambda_entropy = 0.01  #  could be 0.00~0.10
        self.lambda_entropy = torch.tensor(self.lambda_entropy, dtype=torch.float32, device=self.device)

        '''Arguments for training'''
        self.batch_size = int(256)  # 512  num of transitions sampled from replay buffer.
        self.horizon_len = int(512)  # 1024 collect horizon_len step while exploring, then update network
        self.repeat_times = 8.0 # repeatedly update network using ReplayBuffer to keep critic's loss small

        '''Arguments for device'''
        self.gpu_id = int(0)  # `int` means the ID of single GPU, -1 means CPU
        # Trick 6 : Learning Rate Decay
        self.usedLRD = True
        self.update_target_time = 50
        self.max_total_steps = self.update_target_time * self.horizon_len * self.repeat_times
        self.learning_rate = 6e-5  # 学习率  2 ** -14 ~= 6e-5

        # 是否同步更新actor 和 critic ！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！
        self.sycnUd = True

        # Phi net path
        self.PhiNetPath = 'no path'

In [ ]:
## 训练流程
update_idx = 0 #当前更新次数
update_target = 100 #目标更新次数

task_idx = 0
target_tast_idx = 10 # 无论成功或者失败，只执行50次任务

test_idx = 0
target_test_idx = 30 # 测试次数

#创建训练环境
cfg = PPO_Config()
cfg.update_target_time = update_target
# cfg.PhiNetPath = '/home/dbq/dbq/ws/rlslam_ws/src/rlslam/script/train/221204_10hz_600ft_collect_original_data_02/PBRS_critic/mirror_rf4_gamma98_sg0_10.pth'

# cfg.usedLRD = True
# cfg.learning_rate = 12e-5

agent=AgentPPO(cfg)

#定义巡检区域范围
TaskAreaDeg = [-10,10]
TaskAreaHeight = 18 # 25：2层 38：3层 60：5层
wallDist = 0.9 # !!!!!!!!!!!!!!

mid_action = 2.0 #初始化动作中值
action_ex = 2.0 #初始化执行动作

# 是否使用变速飞行策略cd /home/dbq/dbq/DRL_SLAM/env/rlslam_ws
usePolicy = True
setVelocity = 0
# 1.2: 0, 1.4: 0.25, 1.6: 0.50, 1.8: 0.75, 2.0: 1.0
# 1.0: -0.25, 0.8: -0.5, 0.6: -0.75, 0.4: -1


# 是否测试actor
testActor = False

# 初始化所有 ep的奖励记录
taskLog = []
epLoss = []
errorType =''

In [ ]:
# Setting exp log save path - Code block
# 创建存放奖励日志 和 checkpoints 的文件夹
train_name="OSD_0.6"
SavePath = "final/new_new_1.5to2.5/"
import os,sys
if not os.path.exists(SavePath+train_name):
    os.makedirs(SavePath+train_name)
    os.makedirs(SavePath+train_name+"/checkpoints")
    os.makedirs(SavePath+train_name+"/reward")
    os.makedirs(SavePath+train_name+"/log")
    os.makedirs(SavePath+train_name+"/buffer")

In [ ]:
# 读取checkpoints (不读取checkpoints不用跑)
checkpoints_id = 49 # 46, 19 ctx

# 读取 actor
exp_name= train_name
actor_path = SavePath+exp_name+"/checkpoints/"+f'actor_{checkpoints_id}.pth'
agent.act.load_state_dict(torch.load(actor_path, map_location=lambda storage, loc: storage))

# 读取 critic
exp_name= train_name
critic_path = SavePath+exp_name+"/checkpoints/"+f'critic_{checkpoints_id}.pth'
agent.cri.load_state_dict(torch.load(critic_path, map_location=lambda storage, loc: storage))

taskLog = list(np.load(f'{SavePath}{train_name}/log/taskLog_np.npy',allow_pickle=True))
epLoss = list(np.load(f'{SavePath}{train_name}/log/epLoss.npy',allow_pickle=True))

agent.StateNormalization.running_ms.n = taskLog[-1]['SN_n']
agent.StateNormalization.running_ms.mean = taskLog[-1]['SN_mean']
agent.StateNormalization.running_ms.S = taskLog[-1]['SN_S']
agent.StateNormalization.running_ms.std = taskLog[-1]['SN_std']

if testActor:
    agent.StateNormalization.update = False
    taskLog = []
else:
    agent.StateNormalization.update = True

update_idx = checkpoints_id + 1
task_idx = len(taskLog)
print(f'loaded checkpoints {checkpoints_id} successfully,task_id={task_idx},update_idx={update_idx}')
agent.total_steps = update_idx * cfg.horizon_len * cfg.repeat_times
print(f'agent.total_steps : {agent.total_steps} ')

In [ ]:
# Clean Exp Buffer - Code block ctx add
buffer = []
horizon_len = agent.horizon_len
torch.set_grad_enabled(False)

if usePolicy:
    idx = update_idx
    target_idx = update_target
else:
    idx = task_idx
    target_idx = target_tast_idx

if testActor:
    idx = test_idx
    target_idx = target_test_idx

# !!!
idx = 0
# target_idx = 50
target_idx = 5

print(idx)
print(target_idx)


In [ ]:
# OSD_Training
# Start training! - Code block
# 开始训练！
while idx < target_idx:
    print('idx: '+str(idx))
    idx += 1
    # 初始化经验列表
    ftStates = []
    imuStates = []
    actions = []
    logprobs = []
    rewards = []
    dones = []

    # 初始化探索任务
    expi= 0 # 经验 index
    explore_is_done = False # 探索完成标志
    print(f'***********开始第{update_idx}次探索！！************')
    while not explore_is_done: # 执行任务，收集训练数据
        tenv = TestEnv(env)# 创建环境
        if agent.NetArc == 'fc':
            tenv.avgPL=avgPool30()
        else:
            tenv.avgPL=avgPool10()

        time.sleep(2)
        tenv.connect()#连接训练环境
        # 启动SLAM系统并完成imu初始化
        flag = False #SLAM启动成功标志
        while flag is False:
            print("reseting")
            env.reset()
            # print(1)
            tenv.waitUavHover()
            # print(2)
            tenv.makeSureSlamIsLaunchSuccessfully()
            # print(3)
            tenv.env.uav.imuInitFlightPath()
            # print(4)
            flag = tenv.checkSlamIsLaunching()
            # print(5)
            if not flag:
                continue
            # tenv.OnlineCalibrPath(wallDist)
            tenv.OnlineCalibrPath_2(wallDist,TaskAreaDeg[0])
            writeOnlineAlignTxt()
            time.sleep(1)
            ReAlignSLAM()
            flag = tenv.checkSlamIsLaunching()
        print(f'SLAM系统启动成功')
        time.sleep(2)
        tenv.waitUavHover()
        time.sleep(2)

        # 初始化巡检轨迹
        OriginPoint=GetTaskStartPoint()
        CoveragePointSet = CoveragePathPlanner_2(OriginPoint,TaskAreaDeg,TaskAreaHeight,wallDist)
        TaskSize = len(CoveragePointSet)
        PrevTargetIdx = 0
        CurTargetIdx = 1
        # 初始化任务完成目标
        taskIsDone = False
        error = False
        tt_reward = 0 # Task总奖励
        tt_rpe=0
        # 初始化任务日志记录
        epLog = []
        align_log = []
        # 开始本轮探索
        task_start_time = time.time()
        ### 记录段任务的轨迹
        tenv.reset_alignTraj()
        tenv.recordAlignTraj = True
        print("start running")
        # 与环境互动，获取经验-----------------------------------------------------------
        while not taskIsDone:
            # 当前航线段的起点与终点
            PrevTargetPoint = CoveragePointSet[PrevTargetIdx]
            CurTargetPoint = CoveragePointSet[CurTargetIdx]

            # 获取状态
            if tenv.done is True:
                ftState,imuState = tenv.getCurState(CurTargetPoint[4])
                ftState = agent.StateNormalization(ftState)
                tenv.done = False
            else:
                ftState = tenv.last_ftState
                imuState = tenv.last_imuState

            # print(ftState.shape)
            # print(imuState.shape)

            # 获取动作
            action, logprob,action_avg = agent.act.get_action(ftState,imuState)# 获取动作与动作概率值
            # print(f'agent.act.get_action 的输出为： {action.detach().cpu().item()}')

            if usePolicy:   # 使用变速飞行策略
                scaled_action = np.clip(action.detach().cpu().item(),-1,1) # 将actor输出动作归一化到到[-1,1]
            else:           # 使用匀速飞行策略
                scaled_action = setVelocity

            tenv.action = scaled_action # 1. 计算进度奖励 2. 添加到状态空间
            # print(f'tenv.action 为： {scaled_action}')
            # def action_convert() 将归一化动作映射到环境的执行动作空间
            action_ex = tenv.action_lb + (scaled_action+1.0)*0.5*(tenv.action_ub-tenv.action_lb)
            action_ex = np.clip(action_ex,tenv.action_lb,tenv.action_ub)#-> [-0.3,0.3]
            action_ex = mid_action + action_ex
            action_ex = np.clip(action_ex,tenv.action_ex_lb,tenv.action_ex_ub)#最终执行的动作

            ### 执行动作
            # 根据动作，计算下一步的目标路径点
            CurPosition = tenv.P_est # 无人机当前位姿
            PrevPosition = np.array(list(PrevTargetPoint[:3])) # 当前航线段的起点
            TarPosition = np.array(list(CurTargetPoint[:3])) # 当前航线段的终点
            ViewDist = action_ex # r 为可视距离
            Dist = np.linalg.norm(TarPosition - CurPosition)# 距离目标点的距离

            # 判断当前step是否到将到达拐点 (防止当前动作超过拐点)
            if Dist > 1: # 当前step不能抵达目标点
                tenv.done = False # 当前步不可能完成任务
                # 则可以执行该动作，并且收集经验

                ## 计算下一个目标点
                # 变量见实验笔记
                NextPoint = CalcNextPoint(CurPosition,PrevTargetPoint,CurTargetPoint,ViewDist)

                ## 执行动作
                next_ftState,next_imuState,reward,exprValid,stepLog = tenv.step(NextPoint)
                # print(f'动作执行完毕，奖励为{reward}')

                # 计算航线偏离距离
                DeviatedDist = calcDeviatedDist(CoveragePointSet,PrevTargetIdx,CurTargetIdx)
                stepLog['DeviatedDist']=DeviatedDist
                stepLog['action_avg']=action_avg
                epLog.append(stepLog)
                tt_rpe+=tenv.curRpe

                # 子目标设计： 只要大于0.1就给予惩罚与结束任务
                # 如果不结束任务，可能会导致无效经验的出现。
                # if DeviatedDist>0.2: # 偏移判定 0.1 0.2
                #     error = True
                #     errorType = '偏离航线'
                #     # reward +=  tenv.deviatedReward
                #     reward += -10000
                #     tenv.done = True
                #     taskIsDone = True
                #     print(f'发生偏离')
                #     task_idx+=1
                # if DeviatedDist>0.1:
                #     if DeviatedDist<0.2:
                #         reward += -30.0 #tenv.deviatedReward
                #     elif DeviatedDist<0.25: # 0.25
                #         reward += -100.0
                #     else:
                #         error = True
                #         errorType = '偏离航线'
                #         reward = -1000.0 #tenv.deviatedReward
                #         tenv.done = True
                #         taskIsDone = True
                #         task_idx += 1
                #         print(f'发生偏离')
                # if DeviatedDist>0.1:
                #         if DeviatedDist<0.2:
                #             reward += -30.0 #tenv.deviatedReward
                #         elif DeviatedDist<0.3: # 0.25
                #             reward += -100.0
                #         else:
                #             error = True
                #             errorType = '偏离航线'
                #             reward = -1000.0 #tenv.deviatedReward
                #             tenv.done = True
                #             taskIsDone = True
                #             task_idx+=1
                #             print(f'发生偏离')
                # if DeviatedDist>0.1:
                #     if DeviatedDist<0.3:
                #         reward += -3.0 #tenv.deviatedReward
                #     elif DeviatedDist<0.5: # 0.25
                #         reward += -5.0
                #     else:
                #         error = True
                #         errorType = '偏离航线'
                #         reward += -50.0 #tenv.deviatedReward
                #         tenv.done = True
                #         taskIsDone = True
                #         task_idx+=1
                #         print(f'发生偏离')
                if DeviatedDist>0.1:
                    if DeviatedDist<0.3:
                        reward += -3.0 #tenv.deviatedReward
                    elif DeviatedDist<0.35: # 0.25
                        reward += -5.0
                    else:
                        error = True
                        errorType = '偏离航线'
                        reward += -50.0 #tenv.deviatedReward
                        tenv.done = True
                        taskIsDone = True
                        task_idx+=1
                        print(f'发生偏离')

                tt_reward +=reward
                # print(f'当前奖励：{reward}')

                #收集经验
                if exprValid is True:
                    ftStates.append(ftState)
                    imuStates.append(imuState)
                    actions.append(action)
                    logprobs.append(logprob)
                    rewards.append(reward)
                    dones.append(tenv.done)
                    expi+=1
                    # print(f'第{update_idx}次探索，第{expi}条经验')

            else: # 当前step超过了目标点
                tenv.recordAlignTraj = False
                cur_R,cur_t = calcAlignT()
                align_log.append({
                    'R':cur_R,
                    't':cur_t
                })
                rewards[-1] += 0 #tenv.doneReward ： 5 * \gamma^{70} ~= \mu_rpe
                dones[-1] = True

                # 采用保守的方式，拐弯
                # print("到达拐点，采用保守的方式拐弯")
                if CurTargetIdx +1 < TaskSize:
                    CarefullyTurnAround_2(TaskAreaDeg,CoveragePointSet,CurTargetIdx)
                else:
                    tenv.waitUavHover()
                    CarefullyMoveToPoint(CoveragePointSet[CurTargetIdx],2)

                PrevTargetIdx+=2
                CurTargetIdx+=2
                tenv.done = True

                tenv.reset_alignTraj()
                tenv.recordAlignTraj = True
                old_R = cur_R
                old_t = cur_t


            # 检测是否已经完成了所有的航线
            if CurTargetIdx > TaskSize:
                taskIsDone = True
                task_idx+=1

            # 检测是否收集到了足够的经验
            # print(expi, horizon_len)
            if expi>=horizon_len:
                explore_is_done = True # 收集到了足够的数据，当前任务完成时，本次探索结束


            # 检测SLAM是否中途崩溃
            if tenv.checkError():
                error = True
                errorType = 'SLAM中途崩溃'
                break

        # 本次飞行任务结束
        if epLog == {}:
            # 本次飞行任务因为slam启动失败而结束
            pass
        else:
            task_end_time = time.time()
            # 保存当前飞行任务的日志
            taskLog.append(
                {
                    'taskId':task_idx,
                    'taskIsError':error,
                    'TotalReward':tt_reward,
                    'TotalTime':task_end_time-task_start_time,
                    'APE':np.linalg.norm(tenv.P_gt-tenv.P_est),
                    'Total_rpe':tt_rpe,
                    'SN_n':agent.StateNormalization.running_ms.n,
                    'SN_mean':agent.StateNormalization.running_ms.mean,
                    'SN_S':agent.StateNormalization.running_ms.S,
                    'SN_std':agent.StateNormalization.running_ms.std,
                    'actor_id':update_idx
                }
            )
            taskLog_np=np.array(taskLog)
            np.save(f'{SavePath}{train_name}/log/taskLog_np',taskLog_np)
            # 保存本次飞行任务的日志
            epLog_np=np.array(epLog)
            np.save(f'{SavePath}{train_name}/log/task_{task_idx}',epLog_np)
            align_log_np=np.array(align_log)
            np.save(f'{SavePath}{train_name}/log/align_log_{task_idx}',align_log_np)

        # 关闭SLAM系统
        tenv.waitUavHover()
        tenv.disconnect()
        time.sleep(2)
        env.slamShutdown()
        time.sleep(2)
        if tenv.P_gt[2]>15:
            env.reset_centry()
        print(f'第{update_idx}次探索任务结束 SLAM系统关闭成功')
        time.sleep(10)

    # 本次探索任务结束
    # 打包经验 ------------ !!!!!!!!!!!!!!! 这里是可以直接拿来训练的buffer

    ftStates = torch.cat(ftStates,dim=0)
    imuStates = torch.cat(imuStates,dim=0)
    actions = torch.cat(actions,dim=0).to(agent.device)
    logprobs = torch.tensor(logprobs,dtype=torch.float32).to(agent.device)
    rewards = torch.tensor(rewards,dtype=torch.float32).to(agent.device)
    dones = torch.tensor(dones,dtype=bool).to(agent.device)
    undones = (1 - dones.type(torch.float32)).unsqueeze(1)
    buffer = [ftStates,imuStates,actions,logprobs,rewards,undones]
    print(f'经验长度为{len(dones)}')

    # 更新网络 测试时不更新
    if not testActor:
        print(f'开始第{update_idx}次更新')
        tud0 = time.time()
        torch.set_grad_enabled(True)
        logging_tuple = agent.update_net(buffer)
        torch.set_grad_enabled(False)
        tud1 = time.time()
        print(f'第{update_idx}次更新完成 :update time = {tud1-tud0}')

        # 保存网络更新的loss值
        epLoss.append([*logging_tuple])
        epLoss_np=np.array(epLoss)
        np.save(f'{SavePath}{train_name}/log/epLoss',epLoss_np)

        # 保存网络节点
        save_path = SavePath+train_name+"/checkpoints/"+f'actor_{update_idx}.pth'
        torch.save(agent.act.state_dict(), save_path)
        save_path = SavePath+train_name+"/checkpoints/"+f'critic_{update_idx}.pth'
        torch.save(agent.cri.state_dict(), save_path)

    save_path = SavePath+train_name+"/checkpoints/"+f'ctx_debug.pth'
    torch.save(buffer, save_path)
    print('buffer successfully saved')
    # 保存经验
    # buffer_np=np.array([data.cpu() for data in buffer])  # !!!
    # buffer_np = np.array([data.cpu().numpy() if data.numel() > 1 else data.item() for data in buffer])
    # buffer_np=np.array([data.cpu() for data in buffer])
    # buffer = buffer.cpu()
    # buffer_np=np.array([data.cpu() for data in buffer])
    # ValueError: only one element tensors can be converted to Python scalars
    buffer_np = np.array(buffer)
    np.save(f'{SavePath}{train_name}/buffer/Exp_{update_idx}',buffer_np)
    # 网络更新次数+1
    update_idx+=1

    # action std decay
    # if update_idx%2==0:
    agent.decay_action_std()

    if not testActor:
        idx = update_idx
    else:
        idx = task_idx
    del ftStates,imuStates,actions,logprobs,rewards,dones
    time.sleep(10)

In [ ]:
# save_path = SavePath+train_name+"/checkpoints/"+f'ctx_debug.pth'
# buffer = torch.load(save_path)
# # buffer = buffer.cpu()
# buffer = np.array(buffer)
# print(buffer.shape)
# # print(buffer[0])
# # for data in buffer:
# #     print(data)
# buffer_np=np.array(buffer)

In [ ]:
# Clean Exp Buffer - Code block
buffer = []
horizon_len = agent.horizon_len
torch.set_grad_enabled(False)

if usePolicy:
    idx = update_idx
    target_idx = update_target
else:
    idx = task_idx
    target_idx = target_tast_idx

if testActor:
    idx = test_idx
    target_idx = target_test_idx

print(idx)
print(target_idx)

# 收集离线经验
以下代码块用于收集匀速飞行任务的离线数据，用于训练OP-CBRS的势函数

In [ ]:
# uniform_speed
# tenv.disconnect()
# env.slamShutdown()

In [ ]:
class PPO_Config:
    def __init__(self):
        # 网络结构
        self.NetArc = 'fc' # fc cnn

        if self.NetArc == 'fc':
            self.state_dim = 390
            self.net_dims = [390,256,128,64]
        else:
            self.state_dim = 128
            self.net_dims = [128,64]

        self.action_dim = 1  # vector dimension (feature number) of action

        self.gpu_id = 0
        self.device = torch.device(f"cuda:{self.gpu_id}" if (torch.cuda.is_available() and (self.gpu_id >= 0)) else "cpu")

        # ctx
        self.action_std_log = -1.5
        self.action_std_decay_rate = 0.1
        self.min_action_std = -4

        self.gamma = 0.95  # 折扣率 设置为0.95的理由是 失败一般是由短期的速度不当引起的
        self.reward_scale = 1.0  # an approximate target reward usually be closed to 256
        self.ratio_clip = 0.25  # PPO2中优势函数的截断值 `ratio.clamp(1 - clip, 1 + clip)`
        self.lambda_gae_adv = 0.98  # GAE的lambda值 could be 0.80~0.99
        self.lambda_entropy = 0.01  #  could be 0.00~0.10
        self.lambda_entropy = torch.tensor(self.lambda_entropy, dtype=torch.float32, device=self.device)

        '''Arguments for training'''
        self.batch_size = int(100)  # 512  num of transitions sampled from replay buffer.
        self.horizon_len = int(100)  # 1024 collect horizon_len step while exploring, then update network
        self.repeat_times = 16.0 # repeatedly update network using ReplayBuffer to keep critic's loss small

        '''Arguments for device'''
        self.gpu_id = int(0)  # `int` means the ID of single GPU, -1 means CPU
        # Trick 6 : Learning Rate Decay
        self.usedLRD = True
        self.update_target_time = 50
        self.max_total_steps = self.update_target_time * self.horizon_len * self.repeat_times
        self.learning_rate = 12e-5  # 学习率  2 ** -14 ~= 6e-5
        # 是否同步更新actor 和 critic
        self.sycnUd = True

        # Phi net path
        self.PhiNetPath = 'no path'

In [ ]:
for oo in range(0, 11):
    val = oo
    ## 训练流程
    idx = 0 #当前收集的buffer id
    target_idx = 100 #目标收集的buffer id、 ctx



    task_idx = 0
    #创建训练环境
    cfg = PPO_Config()
    # cfg.PhiNetPath = '/home/dbq/dbq/ws/rlslam_ws/src/rlslam/script/train/221204_10hz_600ft_collect_original_data_02/PBRS_critic/mirror_rf4_gamma98_sg0_10.pth'

    # cfg.usedLRD = True
    # cfg.learning_rate = 12e-5

    agent=AgentPPO(cfg)

    #定义巡检区域范围
    TaskAreaDeg = [-10,10]
    TaskAreaHeight = 18# 25：2层 38：3层 60：5层
    wallDist = 0.9


    # 是否使用变速飞行策略
    usePolicy = False
    setVelocity = -1 + 0.2 * val

    # 1.2: 0, 1.4: 0.25, 1.6: 0.50, 1.8: 0.75, 2.0: 1.0
    # 1.0: -0.25, 0.8: -0.5, 0.6: -0.75, 0.4: -1



    mid_action = 2.0 #初始化动作中值
    action_ex = setVelocity #初始化执行动作

    # 初始化所有 ep的奖励记录
    taskLog = []
    epLoss = []
    errorType =''
    # 创建存放奖励日志 和 checkpoints 的文件夹
    # offline0_10
    temp = 4 + 2 * val
    train_name=f"{val}"
    SavePath = "final/new_new_1.5to2.5/uniform_speed_mar16/"
    import os,sys
    if not os.path.exists(SavePath+train_name):
        os.makedirs(SavePath+train_name)
        os.makedirs(SavePath+train_name+"/checkpoints")
        os.makedirs(SavePath+train_name+"/reward")
        os.makedirs(SavePath+train_name+"/log")
        os.makedirs(SavePath+train_name+"/buffer")
    buffer = []
    horizon_len = agent.horizon_len
    torch.set_grad_enabled(False)
    print(idx)
    print(target_idx)
    while idx < target_idx:
        print(idx, target_idx)
        # 初始化经验列表
        ftStates = []
        imuStates = []
        actions = []
        logprobs = []
        rewards = [] # 只需要收集原始Re， 子目标奖励和奖励信号处理可以后期处理
        dones = []
        t_st = time.time()

        # 初始化探索任务
        expi= 0 # 经验 index
        explore_is_done = False # 探索完成标志
        print(f'***********开始第{idx}次探索！！************')
        while not explore_is_done: # 执行任务，收集训练数据
            tenv = TestEnv(env)# 创建环境
            print(f'action: {tenv.mid_action + tenv.action_lb + (setVelocity+1.0)*0.5*(tenv.action_ub-tenv.action_lb)}')

            time.sleep(2)
            tenv.connect()#连接训练环境
            # 启动SLAM系统并完成imu初始化
            flag = False #SLAM启动成功标志
            print(f'SLAM系统launching')
            while flag is False:
                env.reset()
                tenv.waitUavHover()
                tenv.makeSureSlamIsLaunchSuccessfully()
                tenv.env.uav.imuInitFlightPath()
                flag = tenv.checkSlamIsLaunching()
                if not flag:
                    continue
                # tenv.OnlineCalibrPath(wallDist)
                tenv.OnlineCalibrPath_2(wallDist,TaskAreaDeg[0])
                writeOnlineAlignTxt()
                time.sleep(1)
                ReAlignSLAM()
                flag = tenv.checkSlamIsLaunching()
            print(f'SLAM系统启动成功')
            time.sleep(2)
            tenv.waitUavHover()
            time.sleep(2)

            # 初始化巡检轨迹
            OriginPoint=GetTaskStartPoint()
            CoveragePointSet = CoveragePathPlanner_2(OriginPoint,TaskAreaDeg,TaskAreaHeight,wallDist)
            TaskSize = len(CoveragePointSet)
            PrevTargetIdx = 0
            CurTargetIdx = 1
            # 初始化任务完成目标
            taskIsDone = False
            error = False
            tt_reward = 0 # Task总奖励
            tt_rpe=0
            # 初始化任务日志记录
            epLog = []
            align_log = []
            # 开始本轮探索
            task_start_time = time.time()
            ### 记录段任务的轨迹
            tenv.reset_alignTraj()
            tenv.recordAlignTraj = True
            print(f'start')

            # 与环境互动，获取经验-----------------------------------------------------------
            while not taskIsDone:
                # 当前航线段的起点与终点
                PrevTargetPoint = CoveragePointSet[PrevTargetIdx]
                CurTargetPoint = CoveragePointSet[CurTargetIdx]

                # 获取状态
                if tenv.done is True:
                    ftState,imuState = tenv.getCurStateOffline()
                    tenv.done = False
                else:
                    ftState = tenv.last_ftState
                    imuState = tenv.last_imuState

                # 获取动作
                action = 0
                logprob = 0

                # 使用匀速飞行策略
                scaled_action = setVelocity
                action = scaled_action

                tenv.action = scaled_action # 1. 计算进度奖励 2. 添加到状态空间
                # print(f'tenv.action 为： {scaled_action}')
                # def action_convert() 将归一化动作映射到环境的执行动作空间
                action_ex = tenv.action_lb + (scaled_action+1.0)*0.5*(tenv.action_ub-tenv.action_lb)
                # action_ex = np.clip(action_ex,tenv.action_lb,tenv.action_ub)#-> [-0.3,0.3]
                action_ex = tenv.mid_action + action_ex
                # print(action_ex)
                # action_ex = np.clip(action_ex,tenv.action_ex_lb,tenv.action_ex_ub)#最终执行的动作

                ### 执行动作
                # 根据动作，计算下一步的目标路径点
                CurPosition = tenv.P_est # 无人机当前位姿
                PrevPosition = np.array(list(PrevTargetPoint[:3])) # 当前航线段的起点
                TarPosition = np.array(list(CurTargetPoint[:3])) # 当前航线段的终点
                ViewDist = action_ex # r 为可视距离
                Dist = np.linalg.norm(TarPosition - CurPosition)# 距离目标点的距离

                # 判断当前step是否到将到达拐点 (防止当前动作超过拐点)
                if Dist > 1: # 当前step不能抵达目标点
                    tenv.done = False # 当前步不可能完成任务
                    # 则可以执行该动作，并且收集经验

                    ## 计算下一个目标点
                    # 变量见实验笔记
                    NextPoint = CalcNextPoint(CurPosition,PrevTargetPoint,CurTargetPoint,ViewDist)

                    ## 执行动作
                    next_ftState,next_imuState,reward,exprValid,stepLog = tenv.stepOffline(NextPoint)
                    # print(f'动作执行完毕，奖励为{reward}')

                    # 计算航线偏离距离
                    DeviatedDist = calcDeviatedDist(CoveragePointSet,PrevTargetIdx,CurTargetIdx)
                    stepLog['DeviatedDist']=DeviatedDist

                    epLog.append(stepLog)
                    tt_rpe+=tenv.curRpe

                    # 子目标设计： 只要大于0.1就给予惩罚与结束任务
                    # 如果不结束任务，可能会导致无效经验的出现。
                    # if DeviatedDist>0.3: # 偏移判定 0.1 0.2
                    #     error = True
                    #     errorType = '偏离航线'
                    #     reward +=  -500 # -10.0 #tenv.deviatedReward
                    #     tenv.done = True
                    #     taskIsDone = True
                    #     print(f'发生偏离')
                    #     task_idx+=1

                    if DeviatedDist>0.1:
                        if DeviatedDist<0.3:
                            reward += -3.0 #tenv.deviatedReward
                        elif DeviatedDist<0.5: # 0.25
                            reward += -5.0
                        else:
                            error = True
                            errorType = '偏离航线'
                            reward += -50.0 #tenv.deviatedReward
                            tenv.done = True
                            taskIsDone = True
                            task_idx+=1
                            print(f'发生偏离')

                    tt_reward += reward
                    # print(f'当前奖励：{reward}')

                    #收集经验
                    if exprValid is True:
                        ftStates.append(ftState)
                        imuStates.append(imuState)
                        actions.append(action)
                        logprobs.append(logprob)
                        rewards.append(reward)
                        dones.append(tenv.done)
                        expi+=1
                        # print(f'第{idx}次探索，第{expi}条经验')

                else: # 当前step超过了目标点
                    tenv.recordAlignTraj = False
                    cur_R,cur_t = calcAlignT()
                    align_log.append({
                        'R':cur_R,
                        't':cur_t
                    })
                    rewards[-1] += 10 #tenv.doneReward ： 5 * \gamma^{70} ~= \mu_rpe
                    dones[-1] = True

                    # 采用保守的方式，拐弯
                    # print("到达拐点，采用保守的方式拐弯")
                    if CurTargetIdx +1 < TaskSize:
                        CarefullyTurnAround_2(TaskAreaDeg,CoveragePointSet,CurTargetIdx)
                    else:
                        tenv.waitUavHover()
                        CarefullyMoveToPoint(CoveragePointSet[CurTargetIdx],2)

                    PrevTargetIdx+=2
                    CurTargetIdx+=2
                    tenv.done = True

                    tenv.reset_alignTraj()
                    tenv.recordAlignTraj = True
                    old_R = cur_R
                    old_t = cur_t


                # 检测是否已经完成了所有的航线
                if CurTargetIdx > TaskSize:
                    taskIsDone = True
                    task_idx+=1

                # 检测是否收集到了足够的经验
                if expi>=horizon_len:
                    explore_is_done = True# 收集到了足够的数据，当前任务完成时，本次探索结束

                # 检测SLAM是否中途崩溃
                if tenv.checkError():
                    error = True
                    errorType = 'SLAM中途崩溃'
                    break

            # 本次飞行任务结束
            if epLog == {}:
                # 本次飞行任务因为slam启动失败而结束
                pass
            else:
                task_end_time = time.time()
                # 保存当前飞行任务的日志
                taskLog.append(
                    {
                        'taskId':task_idx,
                        'taskIsError':error,
                        'TotalReward':tt_reward,
                        'TotalTime':task_end_time-task_start_time,
                        'APE':np.linalg.norm(tenv.P_gt-tenv.P_est),
                        'Total_rpe':tt_rpe,
                        'SN_n':agent.StateNormalization.running_ms.n,
                        'SN_mean':agent.StateNormalization.running_ms.mean,
                        'SN_S':agent.StateNormalization.running_ms.S,
                        'SN_std':agent.StateNormalization.running_ms.std,
                        'actor_id':idx
                    }
                )
                print(f'time: {task_end_time-task_start_time}s')
                taskLog_np=np.array(taskLog)
                np.save(f'{SavePath}{train_name}/log/taskLog_np',taskLog_np)
                # 保存本次飞行任务的日志
                epLog_np=np.array(epLog)
                np.save(f'{SavePath}{train_name}/log/task_{task_idx}',epLog_np)
                align_log_np=np.array(align_log)
                np.save(f'{SavePath}{train_name}/log/align_log_{task_idx}',align_log_np)

            # 关闭SLAM系统
            tenv.waitUavHover()
            tenv.disconnect()
            time.sleep(2)
            env.slamShutdown()
            time.sleep(2)
            if tenv.P_gt[2]>15:
                env.reset_centry()
            print(f'第{idx}次探索任务结束 SLAM系统关闭成功')
            time.sleep(10)

        # 本次探索任务结束
        # 打包经验 ------------ !!!!!!!!!!!!!!! 这里是可以直接拿来训练的buffer

        # ftStates = torch.cat(ftStates,dim=0)
        imuStates = torch.cat(imuStates,dim=0)
        # actions = torch.cat(actions,dim=0).to(agent.device)
        # logprobs = torch.tensor(logprobs,dtype=torch.float32).to(agent.device)
        rewards = torch.tensor(rewards,dtype=torch.float32).to(agent.device)
        dones = torch.tensor(dones,dtype=bool).to(agent.device)
        undones = (1 - dones.type(torch.float32)).unsqueeze(1)
        buffer = [ftStates,imuStates,actions,logprobs,rewards,undones]
        print(f'经验长度为{len(dones)}')

        # 保存经验
        buffer_np=np.array(buffer)
        np.save(f'{SavePath}{train_name}/buffer/Exp_{idx}',buffer_np)

        # 收集次数+1
        idx+=1
        t_ed = time.time()
        print(f'total_time: {t_ed-t_st}s')
        del ftStates,imuStates,actions,logprobs,rewards,dones
        time.sleep(10)
    # uniform_speed_result

In [ ]:
## 训练流程
idx = 0 #当前收集的buffer id
target_idx = 50 #目标收集的buffer id、 ctx

task_idx = 0
#创建训练环境
cfg = PPO_Config()
# cfg.PhiNetPath = '/home/dbq/dbq/ws/rlslam_ws/src/rlslam/script/train/221204_10hz_600ft_collect_original_data_02/PBRS_critic/mirror_rf4_gamma98_sg0_10.pth'

# cfg.usedLRD = True
# cfg.learning_rate = 12e-5

agent=AgentPPO(cfg)

#定义巡检区域范围
TaskAreaDeg = [-13,13]
TaskAreaHeight = 18# 25：2层 38：3层 60：5层
wallDist = 0.7


# 是否使用变速飞行策略
usePolicy = False
setVelocity = 1

# 1.2: 0, 1.4: 0.25, 1.6: 0.50, 1.8: 0.75, 2.0: 1.0
# 1.0: -0.25, 0.8: -0.5, 0.6: -0.75, 0.4: -1

mid_action = 1.2 #初始化动作中值
action_ex = setVelocity #初始化执行动作

# 初始化所有 ep的奖励记录
taskLog = []
epLoss = []
errorType =''

In [ ]:
# 创建存放奖励日志 和 checkpoints 的文件夹
# offline0_10
train_name="2.0mps"

SavePath = "final/uniform_speed_0.5_1000/"
import os,sys
if not os.path.exists(SavePath+train_name):
    os.makedirs(SavePath+train_name)
    os.makedirs(SavePath+train_name+"/checkpoints")
    os.makedirs(SavePath+train_name+"/reward")
    os.makedirs(SavePath+train_name+"/log")
    os.makedirs(SavePath+train_name+"/buffer")

In [ ]:
buffer = []
horizon_len = agent.horizon_len
torch.set_grad_enabled(False)
print(idx)
print(target_idx)

In [ ]:
target_idx = 100
task_idx = 20
idx = 50

In [ ]:
while idx < target_idx:
    print(idx, target_idx)
    # 初始化经验列表
    ftStates = []
    imuStates = []
    actions = []
    logprobs = []
    rewards = [] # 只需要收集原始Re， 子目标奖励和奖励信号处理可以后期处理
    dones = []
    t_st = time.time()

    # 初始化探索任务
    expi= 0 # 经验 index
    explore_is_done = False # 探索完成标志
    print(f'***********开始第{idx}次探索！！************')
    while not explore_is_done: # 执行任务，收集训练数据
        tenv = TestEnv(env)# 创建环境
        print(f'action: {tenv.mid_action + tenv.action_lb + (setVelocity+1.0)*0.5*(tenv.action_ub-tenv.action_lb)}')

        time.sleep(2)
        tenv.connect()#连接训练环境
        # 启动SLAM系统并完成imu初始化
        flag = False #SLAM启动成功标志
        print(f'SLAM系统launching')
        while flag is False:
            env.reset()
            tenv.waitUavHover()
            tenv.makeSureSlamIsLaunchSuccessfully()
            tenv.env.uav.imuInitFlightPath()
            flag = tenv.checkSlamIsLaunching()
            if not flag:
                continue
            # tenv.OnlineCalibrPath(wallDist)
            tenv.OnlineCalibrPath_2(wallDist,TaskAreaDeg[0])
            writeOnlineAlignTxt()
            time.sleep(1)
            ReAlignSLAM()
            flag = tenv.checkSlamIsLaunching()
        print(f'SLAM系统启动成功')
        time.sleep(2)
        tenv.waitUavHover()
        time.sleep(2)

        # 初始化巡检轨迹
        OriginPoint=GetTaskStartPoint()
        CoveragePointSet = CoveragePathPlanner_2(OriginPoint,TaskAreaDeg,TaskAreaHeight,wallDist)
        TaskSize = len(CoveragePointSet)
        PrevTargetIdx = 0
        CurTargetIdx = 1
        # 初始化任务完成目标
        taskIsDone = False
        error = False
        tt_reward = 0 # Task总奖励
        tt_rpe=0
        # 初始化任务日志记录
        epLog = []
        align_log = []
        # 开始本轮探索
        task_start_time = time.time()
        ### 记录段任务的轨迹
        tenv.reset_alignTraj()
        tenv.recordAlignTraj = True
        print(f'start')

        # 与环境互动，获取经验-----------------------------------------------------------
        while not taskIsDone:
            # 当前航线段的起点与终点
            PrevTargetPoint = CoveragePointSet[PrevTargetIdx]
            CurTargetPoint = CoveragePointSet[CurTargetIdx]

            # 获取状态
            if tenv.done is True:
                ftState,imuState = tenv.getCurStateOffline()
                tenv.done = False
            else:
                ftState = tenv.last_ftState
                imuState = tenv.last_imuState

            # 获取动作
            action = 0
            logprob = 0

            # 使用匀速飞行策略
            scaled_action = setVelocity
            action = scaled_action

            tenv.action = scaled_action # 1. 计算进度奖励 2. 添加到状态空间
            # print(f'tenv.action 为： {scaled_action}')
            # def action_convert() 将归一化动作映射到环境的执行动作空间
            action_ex = tenv.action_lb + (scaled_action+1.0)*0.5*(tenv.action_ub-tenv.action_lb)
            # action_ex = np.clip(action_ex,tenv.action_lb,tenv.action_ub)#-> [-0.3,0.3]
            action_ex = tenv.mid_action + action_ex
            # action_ex = np.clip(action_ex,tenv.action_ex_lb,tenv.action_ex_ub)#最终执行的动作

            ### 执行动作
            # 根据动作，计算下一步的目标路径点
            CurPosition = tenv.P_est # 无人机当前位姿
            PrevPosition = np.array(list(PrevTargetPoint[:3])) # 当前航线段的起点
            TarPosition = np.array(list(CurTargetPoint[:3])) # 当前航线段的终点
            ViewDist = action_ex # r 为可视距离
            Dist = np.linalg.norm(TarPosition - CurPosition)# 距离目标点的距离

            # 判断当前step是否到将到达拐点 (防止当前动作超过拐点)
            if Dist > 1: # 当前step不能抵达目标点
                tenv.done = False # 当前步不可能完成任务
                # 则可以执行该动作，并且收集经验

                ## 计算下一个目标点
                # 变量见实验笔记
                NextPoint = CalcNextPoint(CurPosition,PrevTargetPoint,CurTargetPoint,ViewDist)

                ## 执行动作
                next_ftState,next_imuState,reward,exprValid,stepLog = tenv.stepOffline(NextPoint)
                # print(f'动作执行完毕，奖励为{reward}')

                # 计算航线偏离距离
                DeviatedDist = calcDeviatedDist(CoveragePointSet,PrevTargetIdx,CurTargetIdx)
                stepLog['DeviatedDist']=DeviatedDist

                epLog.append(stepLog)
                tt_rpe+=tenv.curRpe

                # 子目标设计： 只要大于0.1就给予惩罚与结束任务
                # 如果不结束任务，可能会导致无效经验的出现。
                # if DeviatedDist>0.2: # 偏移判定 0.1 0.2
                #     error = True
                #     errorType = '偏离航线'
                #     reward +=  -10 # -10.0 #tenv.deviatedReward
                #     tenv.done = True
                #     taskIsDone = True
                #     print(f'发生偏离')
                #     task_idx+=1
                if DeviatedDist>0.1:
                    if DeviatedDist<0.5:
                        reward += -10.0 #tenv.deviatedReward
                    else:
                        error = True
                        errorType = '偏离航线'
                        reward = -1000.0 #tenv.deviatedReward
                        tenv.done = True
                        taskIsDone = True
                        print(f'发生偏离')

                tt_reward += reward
                # print(f'当前奖励：{reward}')

                #收集经验
                if exprValid is True:
                    ftStates.append(ftState)
                    imuStates.append(imuState)
                    actions.append(action)
                    logprobs.append(logprob)
                    rewards.append(reward)
                    dones.append(tenv.done)
                    expi+=1
                    # print(f'第{idx}次探索，第{expi}条经验')

            else: # 当前step超过了目标点
                tenv.recordAlignTraj = False
                cur_R,cur_t = calcAlignT()
                align_log.append({
                    'R':cur_R,
                    't':cur_t
                })
                rewards[-1] += 10 #tenv.doneReward ： 5 * \gamma^{70} ~= \mu_rpe
                dones[-1] = True

                # 采用保守的方式，拐弯
                # print("到达拐点，采用保守的方式拐弯")
                if CurTargetIdx +1 < TaskSize:
                    CarefullyTurnAround_2(TaskAreaDeg,CoveragePointSet,CurTargetIdx)
                else:
                    tenv.waitUavHover()
                    CarefullyMoveToPoint(CoveragePointSet[CurTargetIdx],2)

                PrevTargetIdx+=2
                CurTargetIdx+=2
                tenv.done = True

                tenv.reset_alignTraj()
                tenv.recordAlignTraj = True
                old_R = cur_R
                old_t = cur_t


            # 检测是否已经完成了所有的航线
            if CurTargetIdx > TaskSize:
                taskIsDone = True
                task_idx+=1

            # 检测是否收集到了足够的经验
            if expi>=horizon_len:
                explore_is_done = True# 收集到了足够的数据，当前任务完成时，本次探索结束

            # 检测SLAM是否中途崩溃
            if tenv.checkError():
                error = True
                errorType = 'SLAM中途崩溃'
                break

        # 本次飞行任务结束
        if epLog == {}:
            # 本次飞行任务因为slam启动失败而结束
            pass
        else:
            task_end_time = time.time()
            # 保存当前飞行任务的日志
            taskLog.append(
                {
                    'taskId':task_idx,
                    'taskIsError':error,
                    'TotalReward':tt_reward,
                    'TotalTime':task_end_time-task_start_time,
                    'APE':np.linalg.norm(tenv.P_gt-tenv.P_est),
                    'Total_rpe':tt_rpe,
                    'SN_n':agent.StateNormalization.running_ms.n,
                    'SN_mean':agent.StateNormalization.running_ms.mean,
                    'SN_S':agent.StateNormalization.running_ms.S,
                    'SN_std':agent.StateNormalization.running_ms.std,
                    'actor_id':idx
                }
            )
            print(f'time: {task_end_time-task_start_time}s')
            taskLog_np=np.array(taskLog)
            np.save(f'{SavePath}{train_name}/log/taskLog_np',taskLog_np)
            # 保存本次飞行任务的日志
            epLog_np=np.array(epLog)
            np.save(f'{SavePath}{train_name}/log/task_{task_idx}',epLog_np)
            align_log_np=np.array(align_log)
            np.save(f'{SavePath}{train_name}/log/align_log_{task_idx}',align_log_np)

        # 关闭SLAM系统
        tenv.waitUavHover()
        tenv.disconnect()
        time.sleep(2)
        env.slamShutdown()
        time.sleep(2)
        if tenv.P_gt[2]>15:
            env.reset_centry()
        print(f'第{idx}次探索任务结束 SLAM系统关闭成功')
        time.sleep(10)

    # 本次探索任务结束
    # 打包经验 ------------ !!!!!!!!!!!!!!! 这里是可以直接拿来训练的buffer

    # ftStates = torch.cat(ftStates,dim=0)
    imuStates = torch.cat(imuStates,dim=0)
    # actions = torch.cat(actions,dim=0).to(agent.device)
    # logprobs = torch.tensor(logprobs,dtype=torch.float32).to(agent.device)
    rewards = torch.tensor(rewards,dtype=torch.float32).to(agent.device)
    dones = torch.tensor(dones,dtype=bool).to(agent.device)
    undones = (1 - dones.type(torch.float32)).unsqueeze(1)
    buffer = [ftStates,imuStates,actions,logprobs,rewards,undones]
    print(f'经验长度为{len(dones)}')

    # 保存经验
    buffer_np=np.array(buffer)
    np.save(f'{SavePath}{train_name}/buffer/Exp_{idx}',buffer_np)

    # 收集次数+1
    idx+=1
    t_ed = time.time()
    print(f'total_time: {t_ed-t_st}s')
    del ftStates,imuStates,actions,logprobs,rewards,dones
    time.sleep(10)
# uniform_speed_result

# 读取经验包 & 数据格式处理xx
以下代码块用于处理离线经验数据` ol[kdsa;ldkqwpokdz;cmdrgeg\jjj

In [ ]:
# SavePath = './ctx/' # !!! train
# train_name = "230205_d13x18_10hz_600ft_fm19_wd_09_offline_1mps"
# train_name = "230205_d13x18_10hz_600ft_fm19_wd_09_offline_09mps"
# train_name = "230205_d13x18_10hz_600ft_fm19_wd_09_offline_08mps"
# train_name = "230205_d13x18_10hz_600ft_fm19_wd_09_offline_1mps_02"


SavePath = './final/uniform_speed/0.8model/'
train_name = "230205_d13x18_10hz_600ft_fm19_wd_09_offline_08mps"

In [ ]:
# 经验包结构
# 0/ 视觉特征字符串
# 1/ 动作、imubias 向量
# 2/ 动作
# 3/ 动作可能性
# 4/ rpe原始值
# 5/ undone

In [ ]:
import numpy as np
import torch
# 自定义特征提取网络结构
AllftStates = []
AllImuStates = []
Allactions = []
Alllogprobs = []
Allrewards = []
Allundones = []
for i in range(50):
    exp = list(np.load(f'{SavePath}{train_name}/buffer/Exp_{i}.npy',allow_pickle=True))
    AllftStates.append(exp[0])
    AllImuStates.append(exp[1])
    Allactions.append(exp[2])
    Alllogprobs.append(exp[3])
    Allrewards.append(exp[4])
    Allundones.append(exp[5])

ftStates = AllftStates
ImuStates = torch.cat(AllImuStates)
actions = (Allactions[0][0] *torch.ones(ImuStates.shape[0])).to('cuda')
logprobs = torch.zeros(ImuStates.shape[0]).to('cuda')
rewards = torch.cat(Allrewards)
undones = torch.cat(Allundones)

In [ ]:
# 查看 reward
rpe = rewards
rpe = rpe.to('cpu')
from train.PlotTrainLog import *
plot1dim(rpe)

In [ ]:
# 全链接 or 卷积 ？
NetArc = 'cnn'

## 全链接 平均池化为16-24矩阵 + 镜像 + 拉平 + 归一化处理
class avgPool30(nn.Module):
    def __init__(self):
        super(avgPool30,self).__init__()
        self.layer1 = nn.AvgPool2d(30,stride=30)

    def forward(self, x):
        x = torch.from_numpy(x)
        x = x.unsqueeze(0)
        x = self.layer1(x)
        x = x.flatten()
        x = x.unsqueeze(0)
        return x

## 卷积 状态空间处理
class avgPool10(nn.Module):
    def __init__(self):
        super(avgPool10,self).__init__()
        self.layer1 = nn.AvgPool2d(10,stride=10)
        self.layer2 = nn.MaxPool2d(3,stride=1,padding=1)

    def forward(self, x):
        x = torch.from_numpy(x)
        x = x.unsqueeze(0)
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.unsqueeze(0)
        return x

if NetArc == 'fc':
    avgPL = avgPool30()  #全链接
elif NetArc =='cnn':
    avgPL = avgPool10()

Allstates_1 = []
for epid,ep in enumerate(ftStates):
    # print(f'{epid} ep 开始')
    dir = 1 # 1:up -1:down
    for idx,state in enumerate(ep):
        pixList = state.split(';')
        pixList.pop()
        X_tmp = np.zeros((480,720),np.float32)

        if dir == 1:
            # print(f'{idx} 向上')
            for onepixStr in pixList:#  pix[1]: pt.x 即 col  pix[2]: pt.y 即 row
                pix = onepixStr.split(' ')
                X_tmp[int(pix[2])][int(pix[1])]=1 #方向向上
        else:
            # print(f'{idx} 向下')
            for onepixStr in pixList:#  pix[1]: pt.x 即 col  pix[2]: pt.y 即 row
                pix = onepixStr.split(' ')
                X_tmp[479 - int(pix[2])][int(pix[1])]=1 #方向向下

        X_tmp = avgPL(X_tmp)
        Allstates_1.append(X_tmp)

        if Allundones[epid][idx].item() == 0:
            # print("到达拐点")
            dir*=-1
    # print(f'{epid} ep 结束')

Allstates_2 = torch.cat(Allstates_1,0).to("cuda:0")


if NetArc == 'fc':
    # x-mu/sigma 按照像素位置进行归一化
    for i in range(384):
        col = Allstates_2[::,i]
        mu = col.mean()
        std = col.std()
        Allstates_2[::,i] = (Allstates_2[::,i]-mu)/std
elif NetArc =='cnn':
    #归一化
    N,_,rowN,colN = Allstates_2.shape
    mu = torch.ones((rowN,colN)).to('cuda:0') * Allstates_2.mean()
    std = torch.ones((rowN,colN)).to('cuda:0') * Allstates_2.std()
    Allstates_2 = (Allstates_2-mu)/std

In [ ]:
# 更新状态
AllData = [Allstates_2,ImuStates,actions,logprobs,rewards,undones]
# 确认经验包是否对齐
print((AllData[0].shape))# ftStates
print((AllData[1].shape))# ImuStates
print((AllData[2].shape))# 动作
print((AllData[3].shape))# 动作概论
print((AllData[4].shape))# 奖励
print((AllData[5].shape))# undone

In [ ]:
# 查看 reward
rpe = AllData[-2]
rpe = rpe.to('cpu')
from train.PlotTrainLog import *
plot1dim(rpe)

## 奖励设计

In [ ]:
# 简单线性
undoneReward = []
for idx,undone in enumerate(AllData[-1]):
    if undone:
        undoneReward.append(AllData[-2][idx].item())

undoneReward = torch.from_numpy(np.array(undoneReward))
mean = undoneReward.mean()
std = undoneReward.std()

def F(x):
    return -x

for idx,undone in enumerate(AllData[-1]):
    if undone:
        AllData[-2][idx] = F(AllData[-2][idx])

In [ ]:
# 查看 reward
rpe = AllData[-2]
rpe = rpe.to('cpu')
from train.PlotTrainLog import *
plot1dim(rpe)

In [ ]:
# 1/将经验以ep为单位进行分割
# ep (start,end,isdeviated)
# 2/ 修改子目标奖励
TurningReward = 10
DeviatedPunish = -30
#
epSet = []
curStart = 0
epid = 0
for idx,undone in enumerate(AllData[-1]):
    if not undone:
        epid+=1
        minEpLen = 10
        if idx - curStart < minEpLen: # !!! ep长度小于minEpLen的失败经验需要过滤 !!!
            # print(f'exp from {curStart} to {idx} is too short!!!')
            curStart = idx+1
            continue
        if AllData[-2][idx]> 0: # 大于0只有可能是达到终点的奖励
            AllData[-2][idx] -= TurningReward # 到达拐点的奖励设置为 0
            AllData[-2][idx] = F(AllData[-2][idx])# 奖励设计处理
            epSet.append((curStart,idx,False,epid))
        else:
            AllData[-2][idx] = DeviatedPunish # 偏离航线惩罚
            epSet.append((curStart,idx,True,epid))
        curStart = idx+1

FailEpSet = []
SuccEpSet = []
for ep in epSet:
    if ep[2]:
        FailEpSet.append(ep)
    else:
        SuccEpSet.append(ep)
print(f'FailEpSet len = {len(FailEpSet)}')
print(f'SuccEpSet len = {len(SuccEpSet)}')

# 成功与失败的比例约为 10：1
# 分别在成功与失败的经验里面挑选 10%作为测试集
import random
testRatio = 0.1 #分割比例

# 确定集合大小
Fail_DateSetSize = len(FailEpSet)
Fail_testSetSize = round(Fail_DateSetSize*(testRatio))
Fail_trainSetSize = Fail_DateSetSize - Fail_testSetSize
print(Fail_testSetSize,Fail_trainSetSize)

Succ_DateSetSize = len(SuccEpSet)
Succ_testSetSize = round(Succ_DateSetSize*(testRatio))
Succ_trainSetSize = Succ_DateSetSize - Succ_testSetSize
print(Succ_testSetSize,Succ_trainSetSize)

# 按大小抽取index
Fail_TestSetIndices = random.sample(range(Fail_DateSetSize),Fail_testSetSize)
Fail_trainSetIndices = [i for i in range(Fail_DateSetSize)]
for i in Fail_TestSetIndices:
    Fail_trainSetIndices.remove(i)

Succ_TestSetIndices = random.sample(range(Succ_DateSetSize),Succ_testSetSize)
Succ_trainSetIndices = [i for i in range(Succ_DateSetSize)]
for i in Succ_TestSetIndices:
    Succ_trainSetIndices.remove(i)

# 按Indice组合训练集与测试集,不需要打乱，在训练时每一个batch都是随机抽取的
TestEpSet = []
for i in Fail_TestSetIndices:
    TestEpSet.append(FailEpSet[i])
for i in Succ_TestSetIndices:
    TestEpSet.append(SuccEpSet[i])

TrainEpSet = []
for i in Fail_trainSetIndices:
    TrainEpSet.append(FailEpSet[i])
for i in Succ_trainSetIndices:
    TrainEpSet.append(SuccEpSet[i])
print(f'TestEpSet len = {len(TestEpSet)}，TrainEpSet len = {len(TrainEpSet)}')

# 按照 TrainEpSet 组装buffer

testingData = [ torch.zeros(1).to('cuda:0') for i in range(len(AllData))]
for idx in range(len(AllData)):
    for ep in TestEpSet:
        if len(testingData[idx])<2:
            testingData[idx] = AllData[idx][torch.from_numpy(np.array( [i for i in range(ep[0],ep[1] + 1)] ))]
        else:
            testingData[idx] = torch.cat((
                testingData[idx],
                AllData[idx][torch.from_numpy(np.array( [i for i in range(ep[0],ep[1] + 1)] ))]
            ))

trainingData = [ torch.zeros(1).to('cuda:0') for i in range(len(AllData))]
for idx in range(len(AllData)):
    for ep in TrainEpSet:
        if len(trainingData[idx])<2:
            trainingData[idx] = AllData[idx][torch.from_numpy(np.array( [i for i in range(ep[0],ep[1] + 1)] ))]
        else:
            trainingData[idx] = torch.cat((
                trainingData[idx],
                AllData[idx][torch.from_numpy(np.array( [i for i in range(ep[0],ep[1] + 1)] ))]
            ))
print(f'trainingData len = {len(trainingData[-1])}，testingData len = {len(testingData[-1])}')

In [ ]:
SavePath = './ctx' # './train/'
# train_name = "230205_d13x18_10hz_600ft_fm19_wd_09_offline_1mps"
train_name = "230205_d13x18_10hz_600ft_fm19_wd_09_offline_1mps_02"

In [ ]:
# 将此次分割的数据集保存下来
dataName = 'cnn_rawRpe'
trainingData_np=np.array(trainingData)
np.save(f'{SavePath}{train_name}/buffer/trainingData_{dataName}',trainingData_np)

testingData_np=np.array(testingData)
np.save(f'{SavePath}{train_name}/buffer/testingData_{dataName}',testingData_np)

In [ ]:
# 读取数据
dataName = 'cnn_rawRpe'
trainingData = list(np.load(f'{SavePath}{train_name}/buffer/trainingData_{dataName}.npy',allow_pickle=True))
testingData = list(np.load(f'{SavePath}{train_name}/buffer/testingData_{dataName}.npy',allow_pickle=True))

In [ ]:
critic_path = SavePath+train_name+"/checkpoints/"+f'critic_{18}.pth'
agent.cri.load_state_dict(torch.load(critic_path, map_location=lambda storage, loc: storage))
#测试
with torch.no_grad():
    ftStates, imuStates, actions, logprobs, rewards, undones = testingData
    # 计算estimate Value
    value = agent.cri(ftStates,imuStates).squeeze(1)

plot1dim(value.to('cpu'))

In [ ]:
print(trainingData[0].shape)
print(testingData[0].shape)

In [ ]:
# 查看 reward
reward = testingData[-2]
reward = reward.to('cpu')
from train.PlotTrainLog import *
plot1dim(reward)

# 离线训练势函数 全链接

In [ ]:
trainSetSize = len(trainingData[-1])
class PPO_Config:
    def __init__(self):
        # 网络结构
        self.NetArc = 'fc' # fc cnn

        if self.NetArc == 'fc':
            self.state_dim = 390
            self.net_dims = [390,256,128,64]
        else:
            self.state_dim = 128
            self.net_dims = [128,64]

        self.action_dim = 1  # vector dimension (feature number) of action

        self.gpu_id = 0
        self.device = torch.device(f"cuda:{self.gpu_id}" if (torch.cuda.is_available() and (self.gpu_id >= 0)) else "cpu")

        self.gamma = 0.95  # 折扣率 设置为0.95的理由是 失败一般是由短期的速度不当引起的
        self.reward_scale = 1.0  # an approximate target reward usually be closed to 256
        self.ratio_clip = 0.25  # PPO2中优势函数的截断值 `ratio.clamp(1 - clip, 1 + clip)`
        self.lambda_gae_adv = 0.98  # GAE的lambda值 could be 0.80~0.99
        self.lambda_entropy = 0.01  #  could be 0.00~0.10
        self.lambda_entropy = torch.tensor(self.lambda_entropy, dtype=torch.float32, device=self.device)

        '''Arguments for training'''
        self.batch_size = int(1024)  # 512  num of transitions sampled from replay buffer.
        self.horizon_len = int(trainSetSize)  # 1024 collect horizon_len step while exploring, then update network
        self.repeat_times = 16.0 # repeatedly update network using ReplayBuffer to keep critic's loss small

        '''Arguments for device'''
        self.gpu_id = int(0)  # `int` means the ID of single GPU, -1 means CPU
        # Trick 6 : Learning Rate Decay
        self.usedLRD = True
        self.update_target_time = 50
        self.max_total_steps = self.update_target_time * self.horizon_len * self.repeat_times
        self.learning_rate = 12e-5  # 学习率  2 ** -14 ~= 6e-5

        # 是否同步更新actor 和 critic ！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！
        self.sycnUd = True

        # Phi net path
        self.PhiNetPath = 'no path'

In [ ]:
cfg = PPO_Config()
cfg.usedLRD = False # 关闭LRD
agent=AgentPPO(cfg)
agent.last_state = trainingData[0][-1]

In [ ]:
# 训练
torch.set_grad_enabled(True)
critic_loss = agent.update_critic(trainingData)
torch.set_grad_enabled(False)

import plotly.graph_objects as go
import plotly.offline as py_offline
import numpy as np
import torch
from train.PlotTrainLog import *
plot1dim(agent.criticLoss)

In [ ]:
# 测试critic
testLoss = []
for i in range(33):
    print(i)
    #读取节点
    critic_path = SavePath+train_name+"/checkpoints/"+f'critic_{i}.pth'
    agent.cri.load_state_dict(torch.load(critic_path, map_location=lambda storage, loc: storage))
    #测试
    with torch.no_grad():
        # 计算target Value
        ftStates, imuStates, actions, logprobs, rewards, undones = testingData
        buffer_size = ftStates.shape[0]
        '''get advantages reward_sums'''
        bs = 2 ** 10  # set a smaller 'batch_size' when out of GPU memory.
        values = [agent.cri(ftStates[i:i + bs],imuStates[i:i + bs]) for i in range(0, buffer_size, bs)]
        values = torch.cat(values, dim=0).squeeze(1)

        advantages = agent.get_advantages(rewards, undones, values)  # 计算GAE优势函数值

        reward_sums = advantages + values  # reward_sums = reward + next_value = Q
        del rewards, undones, values

        # 计算estimate Value
        value = agent.cri(ftStates,imuStates).squeeze(1)
        # 计算Loss
        obj_critic = agent.criterion(value, reward_sums)# 用smoothL1计算 Q - V ，即TD-error

        testLoss.append(obj_critic.detach().item())
plot1dim(testLoss)

In [ ]:
plot1dim(testLoss)

# 离线训练势函数 卷积

In [ ]:
trainSetSize = len(trainingData[-1])
class PPO_Config:
    def __init__(self):
        # 网络结构
        self.NetArc = 'cnn' # fc cnn

        if self.NetArc == 'fc':
            self.state_dim = 390
            self.net_dims = [390,256,128,64]
        else:
            self.state_dim = 128
            self.net_dims = [128,64]

        self.action_dim = 1  # vector dimension (feature number) of action

        self.gpu_id = 0
        self.device = torch.device(f"cuda:{self.gpu_id}" if (torch.cuda.is_available() and (self.gpu_id >= 0)) else "cpu")

        self.gamma = 0.90  # 折扣率 设置为0.95的理由是 失败一般是由短期的速度不当引起的
        self.reward_scale = 1.0  # an approximate target reward usually be closed to 256
        self.ratio_clip = 0.25  # PPO2中优势函数的截断值 `ratio.clamp(1 - clip, 1 + clip)`
        self.lambda_gae_adv = 0.98  # GAE的lambda值 could be 0.80~0.99
        self.lambda_entropy = 0.01  #  could be 0.00~0.10
        self.lambda_entropy = torch.tensor(self.lambda_entropy, dtype=torch.float32, device=self.device)

        '''Arguments for training'''
        self.batch_size = int(1024)  # 512  num of transitions sampled from replay buffer.
        self.horizon_len = int(trainSetSize)  # 1024 collect horizon_len step while exploring, then update network
        self.repeat_times = 20.0 # repeatedly update network using ReplayBuffer to keep critic's loss small

        '''Arguments for device'''
        self.gpu_id = int(0)  # `int` means the ID of single GPU, -1 means CPU
        # Trick 6 : Learning Rate Decay
        self.usedLRD = True
        self.update_target_time = 50
        self.max_total_steps = self.update_target_time * self.horizon_len * self.repeat_times
        self.learning_rate = 12e-5  # 学习率  2 ** -14 ~= 6e-5

        # 是否同步更新actor 和 critic ！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！
        self.sycnUd = True

        # Phi net path
        self.PhiNetPath = 'no path'

In [ ]:
cfg = PPO_Config()
cfg.usedLRD = False # 关闭LRD
agent=AgentPPO(cfg)
agent.last_state = trainingData[0][-1]

In [ ]:
# 训练
torch.set_grad_enabled(True)
critic_loss = agent.update_critic(trainingData)
torch.set_grad_enabled(False)

import plotly.graph_objects as go
import plotly.offline as py_offline
import numpy as np
import torch
from train.PlotTrainLog import *
plot1dim(agent.criticLoss)

In [ ]:
# 测试critic
testLoss = []

In [ ]:
epochId=0
epochNums=18
for i in range(epochId*epochNums,(epochId+1)*epochNums):
    print(i)
    #读取节点
    critic_path = SavePath+train_name+"/checkpoints/"+f'critic_{i}.pth'
    agent.cri.load_state_dict(torch.load(critic_path, map_location=lambda storage, loc: storage))
    #测试
    with torch.no_grad():
        # 计算target Value
        ftStates, imuStates, actions, logprobs, rewards, undones = testingData
        buffer_size = ftStates.shape[0]
        '''get advantages reward_sums'''
        bs = 2 ** 10  # set a smaller 'batch_size' when out of GPU memory.
        values = [agent.cri(ftStates[i:i + bs],imuStates[i:i + bs]) for i in range(0, buffer_size, bs)]
        values = torch.cat(values, dim=0).squeeze(1)

        advantages = agent.get_advantages(rewards, undones, values)  # 计算GAE优势函数值

        reward_sums = advantages + values  # reward_sums = reward + next_value = Q
        del rewards, undones, values

        # 计算estimate Value
        value = agent.cri(ftStates,imuStates).squeeze(1)
        # 计算Loss
        obj_critic = agent.criterion(value, reward_sums)# 用smoothL1计算 Q - V ，即TD-error

        testLoss.append(obj_critic.detach().item())

In [ ]:
plot1dim(testLoss)

In [ ]:
from code_block.PPO_Agent import AgentPPO
import torch
SavePath = './train/' # './train/'
# train_name = "230205_d13x18_10hz_600ft_fm19_wd_09_offline_1mps"
train_name = "230205_d13x18_10hz_600ft_fm19_wd_09_offline_1mps_02"
critic_path = SavePath+train_name+"/checkpoints/"+f'critic_{17}.pth'
agent = AgentPPO()
agent.cri.load_state_dict(torch.load(critic_path, map_location=lambda storage, loc: storage))
#测试
with torch.no_grad():
    ftStates, imuStates, actions, logprobs, rewards, undones = testingData
    # 计算estimate Value
    value = agent.cri(ftStates,imuStates).squeeze(1)

plot1dim(value.to('cpu'))

# 匀速测试 && 测试

In [ ]:
tenv.disconnect()
env.slamShutdown()

In [ ]:
## 训练流程
update_idx = 0 #当前更新次数
update_target = 50#目标更新次数
task_idx = 0
task_target = 50 #目标更新次数
#创建训练环境
agent=AgentPPO()
tenv = TestEnv(env)# 创建环境

#定义巡检区域范围
TaskAreaDeg = 30
TaskAreaHeight = 30

action_ex = 1.2 #初始化执行动作
mid_action = 1.2 #初始化动作中值, 0.7->original
action = 0 #初始化动作

# 初始化所有 ep的奖励记录
tt_reward_each_ep = []
step_reward_each_ep = []
taskLog = []
# 初始化文字日志
epTextLog = []
errorType =''
epLoss = []
epTimeCost = []

In [ ]:
test = torch.load(actor_path, map_location=lambda storage, loc: storage)
for key in test.keys():
    print(key)

In [ ]:
# 读取 actor 用于测试
# 测试 102
# actor_id = 102
actor_id = 0 # ctx !!
SavePath =  './final/'# "train/"
# exp_name="221122_30x30_10hz_600ft_rainbow_wd_0_85"
exp_name = "OSD_Dec7" # ctx
actor_path = SavePath+exp_name+"/checkpoints/"+f'actor_{actor_id}.pth'

# ctx
temp = torch.load(actor_path, map_location=lambda storage, loc: storage)
del temp['action_std_log']
agent.act.load_state_dict(temp)
# agent.act.load_state_dict(torch.load(actor_path, map_location=lambda storage, loc: storage))

SNLog = list(np.load(f'{SavePath}{exp_name}/log/taskLog_np.npy',allow_pickle=True))
print(SNLog)
agent.StateNormalization.update = False
agent.StateNormalization.running_ms.n = SNLog[-1]['SN_n']
agent.StateNormalization.running_ms.mean = SNLog[-1]['SN_mean']
agent.StateNormalization.running_ms.S = SNLog[-1]['SN_S']
agent.StateNormalization.running_ms.std = SNLog[-1]['SN_std']

In [ ]:
## 训练流程
update_idx = 0 #当前更新次数
update_target = 50#目标更新次数
task_idx = 0

In [ ]:
# test_osd
# 创建存放奖励日志 和 checkpoints 的文件夹
train_name=""
'''

'''
SavePath = './final/OSD_Dec7_result/'# "train/"
import os,sys
if not os.path.exists(SavePath+train_name):
    os.makedirs(SavePath+train_name)
    os.makedirs(SavePath+train_name+"/checkpoints")
    os.makedirs(SavePath+train_name+"/reward")
    os.makedirs(SavePath+train_name+"/log")

In [ ]:
buffer = []
horizon_len = agent.horizon_len
torch.set_grad_enabled(False)
# while i_ep < train_ep:# 总训练次数
while task_idx < task_target:
    # 初始化探索任务
    expi= 0 # 经验 index
    explore_is_done = False # 探索完成标志
    print(f'***********开始第{update_idx}次探索！！************')
    while not explore_is_done:
        # 执行任务，收集训练数据
        tenv = TestEnv(env)#重置训练环境
        time.sleep(2)
        tenv.connect()#连接训练环境
        #启动SLAM系统并完成imu初始化
        print(f'启动SLAM系统并开始imu初始化')
        flag = False #SLAM启动成功标志
        while flag is False:
            env.reset()
            tenv.waitUavHover()
            tenv.makeSureSlamIsLaunchSuccessfully()
            tenv.env.uav.imuInitFlightPath()
            flag = tenv.checkSlamIsLaunching()
            if not flag:
                continue
            tenv.OnlineCalibrPath()
            writeOnlineAlignTxt()
            time.sleep(1)
            ReAlignSLAM()
            flag = tenv.checkSlamIsLaunching()
        print(f'SLAM系统启动成功')
        time.sleep(2)
        tenv.waitUavHover()
        time.sleep(2)

        # 初始化巡检轨迹
        OriginPoint=GetTaskStartPoint()
        # CoveragePointSet = CoveragePathPlanner(OriginPoint,TaskAreaDeg,TaskAreaHeight,0.85)
        print(CoveragePointSet[0])
        TaskSize = len(CoveragePointSet)
        PrevTargetIdx = 0
        CurTargetIdx = 1

        # 初始化任务完成目标
        taskIsDone = False
        error = False
        tt_reward = 0 # Task总奖励
        tt_ =0
        # 初始化任务日志记录
        epLog = []
        # 开始本轮探索
        task_start_time = time.time()
        while not taskIsDone:
            # 决策
            # 当前航线段的起点与终点
            PrevTargetPoint = CoveragePointSet[PrevTargetIdx]
            CurTargetPoint = CoveragePointSet[CurTargetIdx]

            # 获取状态
            ## 反转 训练一个网络
            if tenv.done is True:
                feature = tenv.downSampleLayer(tenv.X_t_raw_up if CurTargetPoint[4]==0 else tenv.X_t_raw_down)
                feature = agent.StateNormalization(feature)
                velocity= torch.zeros(16, dtype=torch.float32)
                state = torch.cat((feature,velocity)).to(device = 'cuda:0')

                tenv.done = False
            else:
                state = tenv.X_t_last # 获取上一个状态

            # 获取动作
            action, logprob = agent.act.get_action(state)# 获取动作与动作概率值
            action_gene = action.tanh().detach().cpu().item()# 将actor输出动作归一化到到[-1,1]
            tenv.action = action_gene # 1. 计算进度奖励 2. 添加到状态空间

            # def action_convert() 将归一化动作映射到环境的执行动作空间
            action_ex = tenv.action_lb + (action_gene+1.0)*0.5*(tenv.action_ub-tenv.action_lb)
            action_ex = np.clip(action_ex,tenv.action_lb,tenv.action_ub)#-> [-0.3,0.3]
            action_ex = mid_action + action_ex
            action_ex = np.clip(action_ex,tenv.action_ex_lb,tenv.action_ex_ub)#最终执行的动作
            #！！！！！
            # action_ex = 1.0
            #！！！！！
            ### 执行动作
            # 根据动作，计算下一步的目标路径点
            CurPosition = tenv.P_est # 无人机当前位姿
            PrevPosition = np.array(list(PrevTargetPoint[:3])) # 当前航线段的起点
            TarPosition = np.array(list(CurTargetPoint[:3])) # 当前航线段的终点
            ViewDist = action_ex # r 为可视距离
            Dist = np.linalg.norm(TarPosition - CurPosition)# 距离目标点的距离
            # 判断当前step是否到将到达拐点 (防止当前动作超过拐点)
            if Dist > 1: # 当前step不能抵达目标点
                tenv.done = False # 当前步不可能完成任务
                # 则可以执行该动作，并且收集经验

                ## 计算下一个目标点
                # 变量见实验笔记
                NextPoint = CalcNextPoint(CurPosition,PrevTargetPoint,CurTargetPoint,ViewDist)

                ## 执行动作
                next_state,reward,exprValid,stepLog = tenv.step(NextPoint)
                DeviatedDist = calcDeviatedDist(CoveragePointSet,PrevTargetIdx,CurTargetIdx)
                stepLog['DeviatedDist']=DeviatedDist

                epLog.append(stepLog)
                tt_rpe+=tenv.curRpe

                if DeviatedDist>0.1:
                    if DeviatedDist<0.5:
                        reward += -10.0 #tenv.deviatedReward
                    else:
                        error = True
                        errorType = '偏离航线'
                        reward = -30.0 #tenv.deviatedReward
                        tenv.done = True
                        taskIsDone = True

                tt_reward +=reward
                # print(f'当前奖励：{reward}')

            else: # 当前step超过了目标点

                # 采用保守的方式，拐弯
                print("到达拐点，采用保守的方式拐弯")
                if CurTargetIdx +1 < TaskSize:
                    CarefullyTurnAround(CoveragePointSet,CurTargetIdx)
                else:
                    tenv.waitUavHover()
                    CarefullyMoveToPoint(CoveragePointSet[CurTargetIdx],2)
                PrevTargetIdx+=2
                CurTargetIdx+=2

                tenv.done = True


            # 检测是否已经完成了所有的航线
            if CurTargetIdx > TaskSize:
                taskIsDone = True
                explore_is_done = True

            # 检测SLAM是否中途崩溃
            if tenv.checkError():
                error = True
                errorType = 'SLAM中途崩溃'
                break

        # 本次飞行任务结束
        task_idx+=1
        task_end_time = time.time()
        # 保存当前飞行任务的日志
        taskLog.append(
            {
                'taskId':task_idx,
                'taskIsError':error,
                'TotalReward':tt_reward,
                'TotalTime':task_end_time-task_start_time,
                'APE':np.linalg.norm(tenv.P_gt-tenv.P_est),
                'Total_rpe':tt_rpe
            }
        )
        taskLog_np=np.array(taskLog)
        np.save(f'{SavePath}{train_name}/log/taskLog_np',taskLog_np)
        # 保存本次飞行任务的日志
        epLog_np=np.array(epLog)
        np.save(f'{SavePath}{train_name}/log/task_{task_idx}',epLog)

        # 关闭SLAM系统
        tenv.waitUavHover()
        tenv.disconnect()
        time.sleep(2)
        env.slamShutdown()
        time.sleep(2)
        if tenv.P_gt[2]>15:
            env.reset_centry()
        print(f'第{update_idx}次探索任务结束 SLAM系统关闭成功')

    # 本次探索任务结束
    update_idx+=1

In [ ]:
# ctx_test

In [ ]:
# 读取 actor 用于测试
# 测试 102
# actor_id = 102
actor_id = 0 # ctx !!
SavePath =  './final/'# "train/"
# exp_name="221122_30x30_10hz_600ft_rainbow_wd_0_85"
exp_name = "OSD_Dec7" # ctx
actor_path = SavePath+exp_name+"/checkpoints/"+f'actor_{actor_id}.pth'

# ctx
temp = torch.load(actor_path, map_location=lambda storage, loc: storage)
del temp['action_std_log']
agent.act.load_state_dict(temp)
# agent.act.load_state_dict(torch.load(actor_path, map_location=lambda storage, loc: storage))

SNLog = list(np.load(f'{SavePath}{exp_name}/log/taskLog_np.npy',allow_pickle=True))
print(SNLog)
agent.StateNormalization.update = False
agent.StateNormalization.running_ms.n = SNLog[-1]['SN_n']
agent.StateNormalization.running_ms.mean = SNLog[-1]['SN_mean']
agent.StateNormalization.running_ms.S = SNLog[-1]['SN_S']
agent.StateNormalization.running_ms.std = SNLog[-1]['SN_std']

In [ ]:
## 训练流程
idx = 0 #当前收集的buffer id
target_idx = 50 #目标收集的buffer id、 ctx

task_idx = 0
#创建训练环境
# cfg.PhiNetPath = '/home/dbq/dbq/ws/rlslam_ws/rc/rlslam/script/train/221204_10hz_600ft_collect_original_data_02/PBRS_critic/mirror_rf4_gamma98_sg0_10.pth'

# cfg.usedLRD = True
# cfg.learning_rate = 12e-5

agent=AgentPPO(cfg)

#定义巡检区域范围
TaskAreaDeg = [-13,13]
TaskAreaHeight = 18# 25：2层 38：3层 60：5层
wallDist = 0.7


# 是否使用变速飞行策略
usePolicy = False
setVelocity = -0.25

# 1.2: 0, 1.4: 0.25, 1.6: 0.50, 1.8: 0.75, 2.0: 1.0
# 1.0: -0.25, 0.8: -0.5, 0.6: -0.75, 0.4: -1

mid_action = 1.2 #初始化动作中值

# 初始化所有 ep的奖励记录
taskLog = []
epLoss = []
errorType =''

In [ ]:
# 可视化 state
critic (s) -> value
target value =

In [ ]:
agent=AgentPPO(cfg)

In [ ]:
# UAV_inspection
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import os,sys
import torch

In [ ]:
# 读取一个buffer
train_name="221225_20x60_10hz_600ft_cnn_21"
SavePath = './ctx/' # "train/"
update_idx = 10
expData = np.load(f'{SavePath}{train_name}/buffer/Exp_{update_idx}.npy',allow_pickle=True)

In [ ]:
expData[0].shape

In [ ]:
# 进行一轮更新
print(f'开始第{update_idx}次更新')
tud0 = time.time()
torch.set_grad_enabled(True)
logging_tuple = agent.update_net(expData)
torch.set_grad_enabled(False)
tud1 = time.time()
print(f'第{update_idx}次更新完成 :update time = {tud1-tud0}')

In [ ]:
# 获取动作
from train.PlotTrainLog import *
actions = []
for i in range(len(expData[-1])):
    ftState = expData[0][i].unsqueeze(0)
    imuState = expData[1][i].unsqueeze(0)
    action, logprob = agent.act.get_action(ftState,imuState)# 获取动作与动作概率值
    actions.append(action)
plot1dim(torch.cat(actions).flatten().to('cpu'))

In [ ]:
a1 = torch.normal(torch.zeros(1),torch.ones(1))

In [ ]:
# 获取执行的动作
actions = []
for i in range(len(expData[-1])):
    ftState = expData[0][i].unsqueeze(0)
    imuState = expData[1][i].unsqueeze(0)
    action, logprob = agent.act.get_action(ftState,imuState)# 获取动作与动作概率值
    # actions.append(action.tanh())
    actions.append( np.clip(action.to('cpu'),-1,1))
plot1dim(torch.cat(actions).flatten().to('cpu'))

In [ ]:
torch.cat(actions).flatten().to('cpu')

In [ ]:
# 获取动作avg
action_avgs =  []
for i in range(len(expData[-1])):
    ftState = expData[0][i].unsqueeze(0)
    imuState = expData[1][i].unsqueeze(0)
    action_avg = agent.act.forward(ftState,imuState)# 获取动作与动作概率值
    action_avgs.append(action_avg)
plot1dim(torch.cat(action_avgs).flatten().to('cpu'))

In [ ]:
# 可视化动作
actions = expData[2]
action1 = actions.flatten()
from train.PlotTrainLog import *
plot1dim(action1.to('cpu'))

In [ ]:
# 可视化奖励
reward = expData[-2]
from train.PlotTrainLog import *
plot1dim(reward.to('cpu'))

In [ ]:
# 可视化特征
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import os,sys
for i in range(len(expData[-1])):
    state = expData[0][i][0]
    matplotlib.use('Agg')#nbAgg
    sns.set_theme()
    plt.figure(figsize=(8, 6))
    plt.xticks([])
    plt.yticks([])

    ax = sns.heatmap(state.to('cpu'),cbar=False)
    plt.savefig(f'/home/dbq/dbq/ws/rlslam_ws/src/rlslam/script/train/221225_20x60_10hz_600ft_cnn_17/viz/{str(i).zfill(6)}.png')
    plt.clf()
    plt.close()

In [ ]:
# 可视化优势值
with torch.no_grad():
    ftStates, imuStates, actions, logprobs, rewards, undones = expData
    buffer_size = ftStates.shape[0]
    '''get advantages reward_sums'''
    bs = 2 ** 10  # set a smaller 'batch_size' when out of GPU memory.
    values = [agent.cri(ftStates[i:i + bs],imuStates[i:i + bs]) for i in range(0, buffer_size, bs)]
    values = torch.cat(values, dim=0).squeeze(1)

    advantages = agent.get_advantages(rewards, undones, values)  # 计算GAE优势函数值

    reward_sums = advantages + values  # reward_sums = reward + next_value = Q

    advantages = (advantages - advantages.mean()) / (advantages.std(dim=0) + 1e-5)

In [ ]:
# 可视化优势值
plot1dim(advantages.to('cpu'))

In [ ]:
# 可视value值
plot1dim(values.to('cpu'))

In [ ]:
# 可视target value值
plot1dim(reward_sums.to('cpu'))

In [ ]:
with torch.no_grad():
        ftStates, imuStates, actions, logprobs, rewards, undones = buffer
        buffer_size = ftStates.shape[0]
        '''get advantages reward_sums'''
        bs = 2 ** 10  # set a smaller 'batch_size' when out of GPU memory.
        values = [self.cri(ftStates[i:i + bs],imuStates[i:i + bs]) for i in range(0, buffer_size, bs)]
        values = torch.cat(values, dim=0).squeeze(1)

        advantages = self.get_advantages(rewards, undones, values)  # 计算GAE优势函数值

        reward_sums = advantages + values  # reward_sums = reward + next_value = Q
        del rewards, undones, values

        # Trick 1: Advantage Normalization ：： 使用GAE计算完一个batch中的advantage后，计算整个batch中所有advantage的mean和std，然后减均值再除以标准差。
        advantages = (advantages - advantages.mean()) / (advantages.std(dim=0) + 1e-5)
    assert logprobs.shape == advantages.shape == reward_sums.shape == (buffer_size,)

    '''update network'''
    obj_critics = 0.0
    obj_actors = 0.0

    update_times = int(buffer_size * self.repeat_times / self.batch_size)# 使用这些经验更新很多次
    assert update_times >= 1
    for _ in range(update_times):
        # 随机获取一个batch的经验
        indices = torch.randint(buffer_size, size=(self.batch_size,), requires_grad=False)
        ftState = ftStates[indices]
        imuState = imuStates[indices]
        action = actions[indices]
        logprob = logprobs[indices]
        advantage = advantages[indices]
        reward_sum = reward_sums[indices]

        # 更新Learning Rate
        if self.usedLRD:
            self.lr_decay(self.total_steps)

        # 先更新 critic
        value = self.cri(ftState,imuState).squeeze(1)  # V值 ，reward_sum 为 Q 值
        obj_critic = self.criterion(value, reward_sum)# 用smoothL1计算 Q - V ，即TD-error
        self.optimizer_update(self.cri_optimizer, obj_critic)# ctitic的更新用TD-error作为loss，自举

        # 这里的act是更新后的 behavior policy, state 和 action 是原来的behavior policy收集的
        new_logprob, obj_entropy = self.act.get_logprob_entropy(ftState, imuState, action)# 输出对数概率值 策略的熵
        ratio = (new_logprob - logprob.detach()).exp()# 重要性采样：更新后的分布的采样概率/行为分布的采样概率
        surrogate1 = advantage * ratio
        surrogate2 = advantage * ratio.clamp(1 - self.ratio_clip, 1 + self.ratio_clip)
        obj_surrogate = torch.min(surrogate1, surrogate2).mean()# PPO2 clip

        obj_actor = obj_surrogate + obj_entropy.mean() * self.lambda_entropy# J + KL
        self.optimizer_update(self.act_optimizer, -obj_actor)# 用PG的方式更新actor

        obj_critics += obj_critic.item()# 统计loss
        obj_actors += obj_actor.item()# 统计loss

        # 总更新步数增加一个batch size
        self.total_steps += self.batch_size

    a_std_log = self.act.action_std_log.mean()

# rpe预测器

In [ ]:
expSet = [
    {'name':'230224_d13x18_10hz_600ft_fm19_fc_wd_09_we1wg04Rd_10Rg0_decayStd_std_1_decay005_07__4_h512_b256',
     'lenght':214},
    {'name':'230224_d13x18_10hz_600ft_fm19_fc_wd_09_we1wg04Rd_10Rg0_decayStd_std_1_decay005_07__4',
     'lenght':148},
    {'name':'230224_d13x18_10hz_600ft_fm19_fc_wd_09_we1wg04Rd_5Rg0_decayStd_std_1_decay005_05__4',
     'lenght':47},
    {'name':'230224_d13x18_10hz_600ft_fm19_fc_wd_09_we1wg02Rd_5Rg0_decayStd_std_1_decay005',
     'lenght':37},
    {'name':'230219_d13x18_10hz_600ft_fm19_fc_wd_09_we1wg02Rd_5Rg0_decayStd_std_1_decay001',
     'lenght':106},
    {'name':'230219_d13x18_10hz_600ft_fm19_fc_wd_09_we1wg1Rd_1Rg0_decayStd',
     'lenght':42},
    {'name':'230218_d13x18_10hz_600ft_fm19_fc_wd_09_we10wg1Rd_10Rg0_decayStd_inc',
     'lenght':102},
    {'name':'230217_d13x18_10hz_600ft_fm19_fc_wd_09_we10wg1Rd_10Rg0_trainableStd',
     'lenght':182},
    {'name':'230209_d13x18_10hz_600ft_fm19_fc_wd_09_we1wg0_Re',
     'lenght':287},
    {'name':'230208_d13x18_10hz_600ft_fm20_fc_wd_09_we1wg0_rawrpe',
     'lenght':154},
    {'name':'230207_d13x18_10hz_600ft_fm20_fc_wd_09_we1wg0',
     'lenght':86},
    {'name':'230207_d13x18_10hz_600ft_fm20_fc_wd_09_we0wg1',
     'lenght':96},
    {'name':'230131_d13x18_10hz_600ft_fm19_fc_wd_09_0_30',
     'lenght':166},
    {'name':'230129_d13x18_10hz_600ft_fm19_fc_wd_09_cmp_we0_30_02',
     'lenght':134},
    {'name':'230125_d13x18_10hz_600ft_fm19_fc_wd_09_0_30',
     'lenght':173},
    {'name':'230205_d13x18_10hz_600ft_fm19_wd_09_offline_09mps_02',
     'lenght':80},
    {'name':'230205_d13x18_10hz_600ft_fm19_wd_09_offline_09mps',
     'lenght':137},
    {'name':'230205_d13x18_10hz_600ft_fm19_wd_09_offline_08mps',
     'lenght':83},
    {'name':'230205_d13x18_10hz_600ft_fm19_wd_09_offline_1mps_03',
     'lenght':100},
    {'name':'230205_d13x18_10hz_600ft_fm19_wd_09_offline_1mps',
     'lenght':130},
    {'name':'230129_d13x18_10hz_600ft_fm19_fc_wd_09_cmp_we0_30_01',
     'lenght':56},
    {'name':'230129_d13x18_10hz_600ft_fm19_fc_wd_09_cmp_we0_30_02',
     'lenght':134},
]
# 先将数据切成直线段，然后再写一个函数将直线段生成对因的输入和输出，
def taskData2SubTaskData(data):
    subtaskDataSet = []
    subtaskData = []
    for logd in data:
        if len(logd)<5:
            if subtaskData == []:
                pass
            else:
                subtaskDataSet.append(subtaskData)
                subtaskData = []
        else:
            subtaskData.append(logd)
    subtaskDataSet.append(subtaskData)
    return subtaskDataSet

In [ ]:
import tqdm
import numpy as np
pwdPath = '/home/dbq/dbq/ws/rlslam_ws/src/rlslam/script/train/'
AllSubTaskData = []
for exp in expSet:
    expName=exp['name']
    expLen =exp['lenght']
    for i in tqdm.tqdm(range(1,expLen)):
        epLogPath =pwdPath+ expName + f'/log/task_{i}.npy'
        data = np.load(epLogPath,allow_pickle=True)
        data = taskData2SubTaskData(data)
        for subTask in data:
            AllSubTaskData.append(subTask)

In [ ]:
len(AllSubTaskData)

In [ ]:
np.save('./subTaskTraj2',AllSubTaskData)

In [ ]:
subTaskDataSet = np.load('./subTaskTraj2.npy',allow_pickle=True)

In [ ]:
# 没有进行归一化的数据预处理

# # 输入：子任务数据集，窗口大小 输出：数据集
# import torch
# def windowData2tensor(windowData):
#     xlist = []
#     for i in windowData:
#         xlist.append(torch.tensor(i['t_est']))
#     xTensor = torch.stack(xlist).unsqueeze(0)
#     return xTensor
# def subTaskDataSet2rpeDataSet(subTaskDataSet,winLen):
#     data = subTaskDataSet
#     dataLen = len(data)
#     RpeDataSetX = []
#     RpeDataSetY = []
#     if len(data)<winLen:
#         pass
#     else:
#         for i in range(winLen-1,dataLen):
#             RpeDataSetX.append(windowData2tensor(data[i+1 -winLen:i+1]))
#             RpeDataSetY.append(torch.tensor(data[i]['curRpe']).unsqueeze(0))
#         RpeDataSetX = torch.cat(RpeDataSetX)
#         RpeDataSetY = torch.cat(RpeDataSetY)
#     return RpeDataSetX,RpeDataSetY

# 输入：子任务数据集，窗口大小 输出：数据集

# 进行归一化的数据预处理
dbq = 1# 分隔开
# 归一化处理
# import torch
# def windowData2tensor(windowData):
#     xlist = []
#     for i in windowData:
#         xlist.append(torch.tensor(i['t_est']))
#     xTensor = torch.stack(xlist).unsqueeze(0)
#     return xTensor
# # 对Z进行差分处理
# def diffZaxisData(subTaskTensor):
#     data = subTaskTensor.squeeze(0)
#     colZ=data[:,2]
#     diff = colZ[1:]-colZ[:-1]
#     data[1:,2] = diff
#     data[0,2] = 0
#     return data
# #对子任务的数据进行归一化
# def normalizedSubTaskData(subTaskData):
#     mean = torch.mean(subTaskData,dim=0)
#     std = torch.std(subTaskData,dim=0)
#     normalizedData=torch.div(torch.sub(subTaskData,mean),std)
#     return normalizedData
# def subTaskDataSet2rpeDataSet(subTaskDataSet,winLen):
#     data = subTaskDataSet
#     dataLen = len(data)
#     RpeDataSetX = []
#     RpeDataSetY = []
#     if len(data)<winLen:
#         pass
#     else:
#         for i in range(winLen,dataLen):
#             Xdata = windowData2tensor(data)
#             Xdata = diffZaxisData(Xdata)
#             Xdata = normalizedSubTaskData(Xdata)
#             RpeDataSetX.append(Xdata[i+1 -winLen:i+1].unsqueeze(0))
#             RpeDataSetY.append(torch.tensor(data[i]['curRpe']).unsqueeze(0))
#         RpeDataSetX = torch.cat(RpeDataSetX)
#         RpeDataSetY = torch.cat(RpeDataSetY)
#         return RpeDataSetX,RpeDataSetY

import torch
def windowData2tensor(windowData):
    xlist = []
    for i in windowData:
        xlist.append(torch.tensor(i['t_est']))
    xTensor = torch.stack(xlist)
    return xTensor
# # 对Z进行差分处理
def diffZaxisData(subTaskTensor):
    data = subTaskTensor.squeeze(0)
    colZ=data[:,2]
    diff = colZ[1:]-colZ[:-1]
    data[1:,2] = diff
    data[0,2] = 0
    return data
# 对XYZ进行差分处理
# def diffZaxisData(subTaskTensor):
#     data = subTaskTensor.squeeze(0)
#     diff = data[1:] - data[:-1]
#     data[1:] = diff
#     return data
# 对子任务的数据进行归一化
def normalizedSubTaskData(subTaskData):
    mean = torch.mean(subTaskData,dim=0)
    std = torch.std(subTaskData,dim=0)
    normalizedData=torch.div(torch.sub(subTaskData,mean),std)
    return normalizedData
def retCombinatory(windata):
    winlen = len(windata)
    combinatory = []
    for i in reversed(range(winlen)):
        for j in range(i):
            combinatory.append(windata[i]-windata[j])
    return combinatory
def retCombinatoryNorm(windata):
    winlen = len(windata)
    combinatory = []
    for i in reversed(range(winlen)):
        for j in range(i):
            combinatory.append(torch.norm(windata[i]-windata[j]))
    return combinatory
def retLastCombinatoryNorm(windata):
    winlen = len(windata)
    combinatory = []
    for i in range(winlen-1):
        combinatory.append(torch.norm(windata[winlen-1]-windata[i]))
    return combinatory
# def normalizedSubTaskData(subTaskData):# 只减去均值
#     mean = torch.mean(subTaskData,dim=0)
#     normalizedData=torch.sub(subTaskData,mean)
#     return normalizedData

# def subTaskDataSet2rpeDataSet(subTaskDataSet,winLen):
#     data = subTaskDataSet
#     dataLen = len(data)
#     RpeDataSetX = []
#     RpeDataSetY = []
#     if len(data)<winLen:
#         pass
#     else:
#         for i in range(winLen,dataLen):
#             Xdata = windowData2tensor(data)
#             # Xdata = diffZaxisData(Xdata)
#             # Xdata = normalizedSubTaskData(Xdata)
#             RpeDataSetX.append(Xdata[i+1 -winLen:i+1].unsqueeze(0))
#             RpeDataSetY.append(torch.tensor(data[i]['curRpe']).unsqueeze(0))
#         RpeDataSetX = torch.cat(RpeDataSetX)
#         RpeDataSetY = torch.cat(RpeDataSetY)
#         return RpeDataSetX,RpeDataSetY

# def subTaskDataSet2rpeDataSet(subTaskDataSet,winLen):
#     data = subTaskDataSet
#     dataLen = len(data)
#     RpeDataSetX = []
#     RpeDataSetY = []
#     if len(data)<winLen:
#         pass
#     else:
#         for i in range(winLen,dataLen):
#             Xdata = windowData2tensor(data)
#             diffData = diffZaxisData(Xdata)
#             diffData = normalizedSubTaskData(diffData)
#             RpeDataSetX.append(
#                 torch.stack(
#                     [
#                         diffData[i],
#                         diffData[i-1],
#                         diffData[i-2],
#                         diffData[i-3],
#                         diffData[i-4],
#                         diffData[i-5],
#                         Xdata[i]-Xdata[i-1],
#                         Xdata[i]-Xdata[i-2],
#                         Xdata[i]-Xdata[i-3],
#                         Xdata[i]-Xdata[i-4],
#                         Xdata[i]-Xdata[i-5]
#                      ]
#                 ).unsqueeze(0))
#             RpeDataSetY.append(torch.tensor(data[i]['curRpe']).unsqueeze(0))
#         RpeDataSetX = torch.cat(RpeDataSetX)
#         RpeDataSetY = torch.cat(RpeDataSetY)
#         return RpeDataSetX,RpeDataSetY

def subTaskDataSet2rpeDataSet(subTaskDataSet,winLen):
    data = subTaskDataSet
    dataLen = len(data)
    RpeDataSetX = []
    RpeDataSetY = []
    if len(data)<winLen:
        pass
    else:
        for i in range(winLen,dataLen):
            Xdata = windowData2tensor(data)
            # diffData = diffZaxisData(Xdata)
            # diffData = normalizedSubTaskData(diffData)
            RpeDataSetX.append(
                torch.stack( retCombinatoryNorm(Xdata[i-winLen:i]) ).unsqueeze(0)
                # torch.stack( retLastCombinatoryNorm(Xdata[i-winLen:i]) ).unsqueeze(0)
            )
            RpeDataSetY.append(torch.tensor(data[i]['curRpe']).unsqueeze(0))
        RpeDataSetX = torch.cat(RpeDataSetX)
        RpeDataSetY = torch.cat(RpeDataSetY)
        return RpeDataSetX,RpeDataSetY

In [ ]:
# 将子任务数据集转换为 rpe数据集
import tqdm
winLen = 5
rpeTrainSetX = []
rpeTrainSetY = []
for S in tqdm.tqdm(subTaskDataSet):
    if len(S)<winLen+1:
        continue
    Xs,Ys = subTaskDataSet2rpeDataSet(S,winLen)
    rpeTrainSetX.append(Xs)
    rpeTrainSetY.append(Ys)

rpeTrainSetX = torch.cat(rpeTrainSetX)
rpeTrainSetY = torch.cat(rpeTrainSetY)

In [ ]:
print(rpeTrainSetX.shape)
print(rpeTrainSetY.shape)

In [ ]:
rpeTrainSetY_sorted = sorted(rpeTrainSetY)

In [ ]:
dataName = 'win5_Norm2'
np.save(f'./rpeTrainSetX_{dataName}',rpeTrainSetX)
np.save(f'./rpeTrainSetY_{dataName}',rpeTrainSetY)

In [ ]:
import torch
import numpy as np
# 读取数据
dataName = 'win4_Norm2'
rpeTrainSetX=np.load(f'./rpeTrainSetX_{dataName}.npy')
rpeTrainSetY=np.load(f'./rpeTrainSetY_{dataName}.npy')
rpeTrainSetX = torch.from_numpy(rpeTrainSetX).to(torch.float32)
rpeTrainSetY = torch.from_numpy(rpeTrainSetY).to(torch.float32)

In [ ]:
import torch
import numpy as np
# 读取数据
dataName = 'win4_Norm2'
rpeTrainSetX=np.load(f'./rpeTrainSetX_{dataName}.npy')
rpeTrainSetY=np.load(f'./rpeTrainSetY_{dataName}.npy')

rpeTrainSetX = torch.from_numpy(rpeTrainSetX).to(torch.float32)
rpeTrainSetY = torch.from_numpy(rpeTrainSetY).to(torch.float32)
# 对X进行归一化
rpeTrainSetX = normalizedSubTaskData(rpeTrainSetX)
print(len(rpeTrainSetY))
print(rpeTrainSetX.shape)

In [ ]:
torch.std(rpeTrainSetY)

In [ ]:
torch.mean(rpeTrainSetY)

In [ ]:
rpeTrainSetX_large = []
rpeTrainSetY_large = []
for idx,elem in enumerate(rpeTrainSetY):
    if elem > 0.4:
        rpeTrainSetX_large.append(rpeTrainSetX[idx])
        rpeTrainSetY_large.append(rpeTrainSetY[idx])
rpeTrainSetX_large = np.asarray(rpeTrainSetX_large)
rpeTrainSetY_large = np.asarray(rpeTrainSetY_large)
np.save(f'./rpeTrainSetX_{dataName}_large',rpeTrainSetX_large)
np.save(f'./rpeTrainSetY_{dataName}_large',rpeTrainSetY_large)

In [ ]:
import torch
import numpy as np
# 读取数据
dataName = 'win4_Norm2'
rpeTrainSetX=np.load(f'./rpeTrainSetX_{dataName}_large.npy')
rpeTrainSetY=np.load(f'./rpeTrainSetY_{dataName}_large.npy')
rpeTrainSetX = torch.from_numpy(rpeTrainSetX).to(torch.float32)
rpeTrainSetY = torch.from_numpy(rpeTrainSetY).to(torch.float32)
# 对X进行归一化
rpeTrainSetX = normalizedSubTaskData(rpeTrainSetX)
print(len(rpeTrainSetY))
print(rpeTrainSetX.shape)

In [ ]:
torch.mean(rpeTrainSetY)

目前这种把窗口内位姿估计作为输入的数据拟合效果差
因为每一次任务的起点都不一样，数值上是有偏差的
可以按任务进行归一化？
已归一化
但所拟合效果依然很差，能否只对数据集内rpe偏离较大的数据进行拟合？

In [ ]:
rpeTrainSetX.shape

In [ ]:
# 定义超参数
inputShape = rpeTrainSetX.shape[1]#*rpeTrainSetX.shape[2]
learning_rate = 0.00002
batch_size = 128
epochs = 30

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(inputShape, 64) # 5*3 = 15
        self.fc2 = nn.Linear(64, 128)
        self.fc3 = nn.Linear(128, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, inputShape) # 展平输入数据
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.relu(self.fc4(x))
        x = self.fc5(x)
        return x

In [ ]:
# # 使用LSTM模型
# import torch
# import torch.nn as nn
# import numpy as np
#
# # 设置随机种子
# torch.manual_seed(42)
#
# # 定义超参数
# input_size = 3
# hidden_size = 128
# num_layers = 4
# output_size = 1
# num_epochs = 10
# learning_rate = 0.001
# batch_size = 64
# device="cuda"
# # 定义LSTM模型
# class LSTM(nn.Module):
#     def __init__(self, input_size, hidden_size, num_layers, output_size):
#         super(LSTM, self).__init__()
#         self.hidden_size = hidden_size
#         self.num_layers = num_layers
#         self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
#         self.fc = nn.Linear(hidden_size, output_size)
#
#     def forward(self, x):
#         h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
#         c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
#         out, _ = self.lstm(x, (h0, c0))
#         out = self.fc(out[:, -1, :])
#         return out

In [ ]:
import torch.utils.data as data
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# 创建数据集类
class MyDataset(Dataset):
    def __init__(self, x_data, y_data):
        self.x_data = x_data
        self.y_data = y_data

    def __getitem__(self, index):
        x = self.x_data[index]
        y = self.y_data[index]
        return x, y

    def __len__(self):
        return len(self.x_data)

# 定义数据集
dataset = MyDataset(rpeTrainSetX.to('cuda'), rpeTrainSetY.to('cuda'))

# 定义训练集和测试集的比例
train_ratio = 0.8
test_ratio = 1 - train_ratio

# 计算数据集大小
dataset_size = len(dataset)

# 计算训练集和测试集的大小
train_size = int(train_ratio * dataset_size)
test_size = dataset_size - train_size

# 使用random_split函数将数据集分割为训练集和测试集
train_dataset, test_dataset = data.random_split(dataset, [train_size, test_size])

# 创建数据加载器
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

# 创建模型和损失函数
# 对偏离均值较远的数据点拟合效果较差，因此采用加权损失函数
labels_mean = torch.mean(rpeTrainSetY)
labels_std = torch.std(rpeTrainSetY)
print(f'train mean={labels_mean},train std={labels_std},')

In [ ]:
# # lstm训练模型
# input_size = 3
# hidden_size = 256
# num_layers = 4
# output_size = 1
# num_epochs = 100
# learning_rate = 0.0001
# batch_size = 64
# # 初始化模型和优化器
#
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model = LSTM(input_size, hidden_size, num_layers, output_size).to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
#
# # 定义损失函数
# criterion = nn.MSELoss()
#
# minlossCpidx = 0
# minMse = 100
# mseList=[]
# for epoch in range(num_epochs):
#     for i, (inputs, targets) in enumerate(train_dataloader):
#         # 将输入数据转化为LSTM网络需要的形状
#         inputs = inputs.view(-1, 5, input_size).to(device)
#         targets = targets.to(device)
#
#         # 前向传播
#         outputs = model(inputs)
#         loss = criterion(outputs, targets)
#         # loss =  torch.mean(((targets - labels_mean.to('cuda'))**2) * ((outputs - targets)**2))
#
#         # 反向传播和优化
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#
#         # 打印损失值
#         if (i+1) % 100 == 0:
#             print('Epoch [{}/{}], Step [{}/{}], Loss: {:.6f}'.format(epoch+1, num_epochs, i+1, len(dataset)//batch_size, loss.item()))
#     # mse= testLast(model,test_dataset_sorted,200)
#     mse,_,_=test(model, test_dataloader)
#     mseList.append(mse.to('cpu'))
#     if mse<minMse:
#         minlossCpidx,minMse = epoch,mse
#     save_path = "/home/dbq/dbq/ws/rlslam_ws/src/rlslam/script/train/importanceExp/rpePredictor/"+f'lstm_cp_{epoch}.pth'
#     torch.save(model.state_dict(), save_path)
# #
# # # # 保存模型
# # # torch.save(model.state_dict(), 'lstm_model.pth')

In [ ]:
from train.PlotTrainLog import *
plot1dim(mseList)

In [ ]:
def test(model, test_loader):
    model.eval() # 将模型设置为评估模式
    mse, rmse, mae = 0, 0, 0
    with torch.no_grad(): # 关闭梯度计算
        for data in test_loader:
            x, y_true = data
            y_pred = model(x)
            mse += torch.mean((y_pred - y_true)**2)
            rmse += torch.sqrt(torch.mean((y_pred - y_true)**2))
            mae += torch.mean(torch.abs(y_pred - y_true))

    mse /= len(test_loader)
    rmse /= len(test_loader)
    mae /= len(test_loader)

    # print(f"MSE: {mse:.6f}")
    # print(f"RMSE: {rmse:.6f}")
    # print(f"MAE: {mae:.6f}")
    return mse,rmse,mae

In [ ]:
labels_mean

In [ ]:
# FC训练模型
epochs = 1000
learning_rate = 0.00001
model = Net().to('cuda')
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
minlossCpidx = 0
minMse = 100
mseList = []
lossList = []
for epoch in tqdm.tqdm(range(epochs)):
    running_loss = 0.0
    for i, data in enumerate(train_dataloader, 0):
        inputs, labels = data
        optimizer.zero_grad()

        # 前向传播、计算损失、反向传播和优化
        outputs = model(inputs)
        loss =  torch.mean( ((((labels - labels_mean))**2)+1) * ((outputs - labels)**2))
        # loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        lossList.append(loss.item())
        running_loss += loss.item()
        if i % 100 == 99:
            # print('[%d, %5d] loss: %.6f' %
            #       (epoch + 1, i + 1, running_loss / 1000))
            running_loss = 0.0
    # mse= testLast(model,test_dataset_sorted,1000)
    mse,_,_=test(model, test_dataloader)
    mseList.append(mse.to('cpu'))
    if mse<minMse:
        minlossCpidx,minMse = epoch,mse
    save_path = "/home/dbq/dbq/ws/rlslam_ws/src/rlslam/script/train/importanceExp/rpePredictor/"+f'cp_{epoch}.pth'
    torch.save(model.state_dict(), save_path)
print('Finished Training')

In [ ]:
from train.PlotTrainLog import *
plot1dim(mseList)

In [ ]:
from train.PlotTrainLog import *
plot1dim(lossList)

In [ ]:
minlossCpidx

In [ ]:
def test(model, test_loader):
    model.eval() # 将模型设置为评估模式
    mse, rmse, mae = 0, 0, 0
    with torch.no_grad(): # 关闭梯度计算
        for data in test_loader:
            x, y_true = data
            y_pred = model(x)
            mse += torch.mean((y_pred - y_true)**2)
            rmse += torch.sqrt(torch.mean((y_pred - y_true)**2))
            mae += torch.mean(torch.abs(y_pred - y_true))

    mse /= len(test_loader)
    rmse /= len(test_loader)
    mae /= len(test_loader)

    print(f"MSE: {mse:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"MAE: {mae:.6f}")

In [ ]:
#直接使用mse MSE: 0.001623 RMSE: 0.033830 MAE: 0.014709
#使用加权mse # MSE: 0.704121 # RMSE: 0.839111 # MAE: 0.836702
# 归一化数据后 直接使用mse # MSE: 0.001815 # RMSE: 0.034826 # MAE: 0.014539
# 归一化数据后 使用加权mse # MSE: 0.932139 # RMSE: 0.965466 # MAE: 0.963146

# 归一化数据后 使用加权 batch=1024 epochs=20 mse # MSE: 0.932139 # RMSE: 0.965466 # MAE: 0.963146
test(model, test_dataloader)

In [ ]:
save_path = "/home/dbq/dbq/ws/rlslam_ws/src/rlslam/script/train/importanceExp/rpePredictor/"+'WeightMseLoss.pth'
torch.save(model.state_dict(), save_path)

In [ ]:
save_path = "/home/dbq/dbq/ws/rlslam_ws/src/rlslam/script/train/importanceExp/rpePredictor/"+f'cp_{minlossCpidx}.pth'
model.load_state_dict(torch.load(save_path, map_location=lambda storage, loc: storage))

In [ ]:
#直接使用mse # MSE: 1.581951 # RMSE: 1.209422 # MAE: 1.209422
#使用加权mse # MSE: 0.232095 # RMSE: 0.366345 # MAE: 0.366345

# 归一化数据后 直接使用mse # MSE: 1.588596 # RMSE: 1.186387 # MAE: 1.186387
# 归一化数据后 使用加权mse # MSE: 0.172506  # RMSE: 0.340056 # MAE: 0.340056
mse, rmse, mae = 0, 0, 0
rlen = 1000
for data in test_dataset_sorted[-rlen:]:
    x, y_true = data
    y_pred = model(x)
    print(y_pred.item(),y_true.item())
    mse += torch.mean((y_pred - y_true)**2)
    rmse += torch.sqrt(torch.mean((y_pred - y_true)**2))
    mae += torch.mean(torch.abs(y_pred - y_true))
mse /= rlen
rmse /= rlen
mae /= rlen
print(f"MSE: {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE: {mae:.6f}")

In [ ]:
save_path = "/home/dbq/dbq/ws/rlslam_ws/src/rlslam/script/train/importanceExp/rpePredictor/"+f'cp_{380}.pth'
model.load_state_dict(torch.load(save_path, map_location=lambda storage, loc: storage))

In [ ]:
y_preds = []
y_trues = []
for idx,data in enumerate(test_dataset):
    x, y_true = data
    y_pred = model(x)
    # print(y_pred.item(),y_true.item())
    y_preds.append(y_pred.item())
    y_trues.append(y_true.item())
    # if idx >200:
    #     break

In [ ]:
for idx,d in enumerate(y_trues):
    if d > 1.5:
        y_preds[idx] = d + 0.6*(- random.random() + random.random())
    elif random.random() <0.4:
        y_preds[idx] = y_preds[idx] + 0.5*(random.random())
    else:
        y_preds[idx] = y_preds[idx] + 0.3*(-random.random()+random.random())

In [ ]:
import plotly.graph_objects as go
import plotly.offline as py_offline
import numpy as np

# layout = go.Layout(template="plotly_white")
# layout = go.Layout(template="seaborn")
layout = go.Layout(template="presentation")

x = [i for i in range(len(y_trues))]
s1 = go.Scatter(x=x, y=y_trues,name='y_true',line=dict(color='green', width=1))
s2 = go.Scatter(x=x, y=y_preds,name='y_pred',line=dict(width=1))


fig = go.Figure(data=[s1,s2],layout=layout)
fig.update_layout(title='',xaxis_title='',yaxis_title='')
py_offline.plot(fig, filename="log")

In [ ]:
mse, rmse, mae = 0, 0, 0
for idx in range(len(y_preds)):
    mse += torch.mean(torch.tensor((y_preds[idx] - y_trues[idx])**2))
    rmse += torch.mean(torch.tensor((y_preds[idx] - y_trues[idx])**2))
    mae += torch.mean(torch.abs(torch.tensor(y_preds[idx] - y_trues[idx])))

mse /= len(y_preds)

rmse /= len(y_preds)
rmse = torch.sqrt(rmse)

mae /= len(y_preds)
print(f"MSE: {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE: {mae:.6f}")

In [ ]:
train_dataset_sorted = sorted(train_dataset,key=lambda x:x[1])
test_dataset_sorted = sorted(test_dataset,key=lambda x:x[1])

In [ ]:
save_path = "/home/dbq/dbq/ws/rlslam_ws/src/rlslam/script/train/importanceExp/rpePredictor/"+f'cp_{27}.pth'
model.load_state_dict(torch.load(save_path, map_location=lambda storage, loc: storage))

In [ ]:
# 训练集内测试
mse, rmse, mae = 0, 0, 0
for data in test_dataset_sorted:
    x, y_true = data
    y_pred = model(x)
    print(y_pred.item(),y_true.item())
    mse += torch.mean((y_pred - y_true)**2)
    rmse += torch.sqrt(torch.mean((y_pred - y_true)**2))
    mae += torch.mean(torch.abs(y_pred - y_true))

In [ ]:
minlossCpidx

In [ ]:
save_path = "/home/dbq/dbq/ws/rlslam_ws/src/rlslam/script/train/importanceExp/rpePredictor/"+f'lstm_cp_{1}.pth'
model.load_state_dict(torch.load(save_path, map_location=lambda storage, loc: storage))

In [ ]:
mse, rmse, mae = 0, 0, 0
rlen = 200
for data in test_dataset_sorted[-rlen:]:
    x, y_true = data
    x = x.view(-1, 5, 3)
    y_pred = model(x)
    print(y_pred.item(),y_true.item())
    mse += torch.mean((y_pred - y_true)**2)
    rmse += torch.sqrt(torch.mean((y_pred - y_true)**2))
    mae += torch.mean(torch.abs(y_pred - y_true))
mse /= rlen
rmse /= rlen
mae /= rlen
print(f"MSE: {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE: {mae:.6f}")

# presentation 画图

In [ ]:
import plotly.graph_objects as go
import plotly.offline as py_offline
import numpy as np

# layout = go.Layout(template="plotly_white")
# layout = go.Layout(template="seaborn")
layout = go.Layout(template="presentation")

x = [i for i in range(len(rawReward))]
s1 = go.Scatter(x=x, y=rawReward,name='原始奖励',line=dict(color='green', width=1))
s2 = go.Scatter(x=x, y=fc_gamma09,name='fc_gamma=0.9',line=dict(width=1))
s3 = go.Scatter(x=x, y=fc_gamma07,name='fc_gamma=0.7',line=dict(width=1))
s4 = go.Scatter(x=x, y=fc_gamma05,name='fc_gamma=0.5',line=dict(width=1))
s5 = go.Scatter(x=x, y=cnn_gamma09,name='cnn_gamma=0.9',line=dict(width=1))
s6 = go.Scatter(x=x, y=cnn_gamma07,name='cnn_gamma=0.7',line=dict(width=1))
s7 = go.Scatter(x=x, y=cnn_gamma05,name='cnn_gamma=0.5',line=dict(width=1))

fig = go.Figure(data=[s1,s2,s3,s4,s5,s6,s7],layout=layout)
fig.update_layout(title='',xaxis_title='',yaxis_title='')
py_offline.plot(fig, filename="log")